# Employee Attrition & HR Analytics
## Python Exploratory Data Analysis

**Author:** Hariom Dubey  
**Project stack:** Python (EDA & statistics) · SQL (querying & validation) · Power BI (dashboarding)

---

### Business Problem

Employee attrition is expensive. Replacing an employee costs the organisation recruitment spend,
onboarding time, lost institutional knowledge and a temporary productivity gap in the team.
HR leadership in this organisation can see *that* people are leaving, but not *which segments of
the workforce* are leaving disproportionately, or *which measurable employee-experience factors*
those exits are associated with.

### Project Objective

Analyse employee-level HR data to describe workforce composition and to identify which
demographic, compensation, satisfaction, career-progression and experience factors are
**associated** with higher attrition, so that HR can target its retention efforts and its
follow-up investigations with evidence rather than intuition.

### Analytical Goals

1. Profile the workforce: departments, job roles, demographics, education, job levels.
2. Quantify attrition overall and across every meaningful employee segment.
3. Distinguish rigorously between **attrition count** and **attrition rate** at all times.
4. Examine compensation, job satisfaction, work-life balance, performance, career progression,
   experience and training against attrition outcomes.
5. Validate the strongest observed patterns with formal hypothesis tests.
6. Produce a transparent, rule-based **HR Risk Segmentation** aligned with the existing SQL logic.
7. Translate findings into HR considerations expressed with appropriate analytical caution.

### Analytical Workflow

```
Explore  →  Clean  →  Transform  →  Analyse  →  Visualise  →  Validate  →  Generate Insights
```

### Role of Each Tool in This Project

| Tool | Responsibility in this project |
|---|---|
| **Python** (this notebook) | Exploration, data-quality auditing, feature engineering, visualisation, statistical testing, business interpretation |
| **SQL** (30 queries) | Structured data extraction, aggregation, window functions, CTEs, metric definition and validation |
| **Power BI** | Interactive dashboard and business communication to non-technical stakeholders |

Python is deliberately **not** used here to re-run all 30 SQL queries. SQL already answers the
structured "what is the number" questions. Python's job in this project is the layer SQL is poor
at: distribution shape, multivariate interaction, statistical validation and visual evidence.

### Dataset

`HR_Analytics_Cleaned_Master.csv` — one row per employee, covering demographics, compensation,
satisfaction ratings, performance, tenure and training. The dataset has already passed through a
cleaning stage prior to this notebook; this notebook **re-verifies** that cleaning independently
rather than assuming it.

### A Note on Analytical Claims

This notebook is an **observational, descriptive analysis**. Nothing in it establishes causation.
Where a relationship is found, it is reported as an *association*. The HR Risk Score in Section 19
is a transparent business rule, **not** a machine-learning prediction, and it does not claim that
any individual will leave.

---

## Business Questions

This notebook is organised around nine questions. Each is answered with evidence in the sections
noted, and every answer is reported as an observed association, never as a causal claim.

1. What is the overall employee attrition rate, and how many employees does it represent? *(Section 7)*
2. Which departments and job roles show meaningfully different attrition patterns — by rate, not just by count? *(Section 7)*
3. How does compensation relate to attrition, and does the compensation data support that comparison at all? *(Section 8)*
4. How are job satisfaction and work-life balance associated with attrition, individually and combined? *(Section 9)*
5. How do age, career experience and company tenure relate to attrition? *(Sections 7, 12)*
6. Are career-progression variables — promotion gap, manager tenure — associated with attrition? *(Section 11)*
7. How do performance rating and training relate to workforce outcomes? *(Sections 10, 13)*
8. Which employee segments meet the predefined HR risk criteria, and how well does that rule separate the workforce? *(Section 18)*
9. Which findings are statistically robust and practically worth HR's attention, and which should be set aside? *(Section 19)*

---

# 1. Import Libraries & Configure Environment

Only the libraries genuinely required for this analysis are imported: `pandas`/`numpy` for data
handling, `matplotlib`/`seaborn` for static charts that render reliably on GitHub, `plotly` for a
single hierarchical chart where interactivity adds real value, and `scipy.stats` for hypothesis
testing.

In [1]:
# Core data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

# Statistics
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

# ---------- Display configuration ----------
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# ---------- Plot configuration ----------
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Consistent project palette: neutral for volume, red reserved for attrition/risk
NEUTRAL, ACCENT, POSITIVE = "#2E4A7D", "#C0392B", "#1E8449"
ATTRITION_PALETTE = {"No": NEUTRAL, "Yes": ACCENT}

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment ready.")
print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

Environment ready.
pandas 2.3.3 | numpy 2.3.5 | seaborn 0.13.2


# 2. Data Loading & Dataset Profile

**Objective:** load the HR dataset, confirm its structure, and build a compact profile table before
any analysis is attempted. The file is located dynamically rather than hard-coded, so the notebook
runs from a `data/` folder, the notebook directory, or the original upload path.

In [2]:
from pathlib import Path

# To point this notebook at a different file, set DATASET_OVERRIDE to its path
# (e.g. DATASET_OVERRIDE = "data/my_export.csv") and re-run. Leave as None to auto-detect.
DATASET_OVERRIDE = None

def locate_dataset(filename_hint="HR_Analytics", extension=".csv"):
    # Search the usual project locations for the HR dataset and return the first match.
    # Keeps the notebook runnable on GitHub, locally, or in the original environment.
    if DATASET_OVERRIDE:
        return Path(DATASET_OVERRIDE)
    search_dirs = [Path("."), Path("data"), Path(".."), Path("/mnt/user-data/uploads")]
    for directory in search_dirs:
        if not directory.exists():
            continue
        matches = sorted(p for p in directory.glob(f"*{extension}") if filename_hint.lower() in p.name.lower())
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"No file matching '*{filename_hint}*{extension}' found in: "
        + ", ".join(str(d) for d in search_dirs)
        + ". Set DATASET_OVERRIDE to the exact file path instead."
    )

DATA_PATH = locate_dataset()
df_raw = pd.read_csv(DATA_PATH)

print(f"Loaded : {DATA_PATH}")
print(f"Shape  : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

FileNotFoundError: No file matching '*HR_Analytics*.csv' found in: ., data, .., \mnt\user-data\uploads. Set DATASET_OVERRIDE to the exact file path instead.

In [ ]:
# First and last records — confirms the file read correctly at both ends
display(df_raw.head())
display(df_raw.tail())

In [ ]:
# Column inventory and data types
schema = pd.DataFrame({
    "Column": df_raw.columns,
    "Dtype": df_raw.dtypes.astype(str).values,
    "Non_Null": df_raw.notna().sum().values,
    "Unique_Values": df_raw.nunique().values,
    "Sample_Value": [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else None for c in df_raw.columns],
})
display(schema)

In [ ]:
# ---------- Separate numerical and categorical variables ----------
# Note: several HR fields are numerically coded ordinal scales (1-4 / 1-5), not true continuous
# measures. They are separated out so they are never averaged or correlated carelessly.
ORDINAL_SCALES = [
    "Education", "JobLevel", "StockOptionLevel", "EnvironmentSatisfaction",
    "JobSatisfaction", "WorkLifeBalance", "JobInvolvement", "PerformanceRating",
]
ID_COLUMNS = ["EmployeeID"]

numeric_cols = [c for c in df_raw.select_dtypes(include=np.number).columns
                if c not in ORDINAL_SCALES + ID_COLUMNS]
categorical_cols = df_raw.select_dtypes(include="object").columns.tolist()

print(f"Continuous numeric variables ({len(numeric_cols)}):\n  {numeric_cols}\n")
print(f"Ordinal (coded) variables ({len(ORDINAL_SCALES)}):\n  {ORDINAL_SCALES}\n")
print(f"Categorical variables ({len(categorical_cols)}):\n  {categorical_cols}")

In [ ]:
# ---------- Compact dataset profile ----------
profile = pd.DataFrame({
    "Metric": [
        "Rows (employees)", "Columns", "Continuous numeric columns", "Ordinal coded columns",
        "Categorical columns", "Identifier columns", "Total missing values",
        "Columns with missing values", "Fully duplicated rows", "Duplicate EmployeeIDs",
        "Memory footprint (MB)",
    ],
    "Value": [
        f"{df_raw.shape[0]:,}", df_raw.shape[1], len(numeric_cols), len(ORDINAL_SCALES),
        len(categorical_cols), len(ID_COLUMNS), f"{int(df_raw.isna().sum().sum()):,}",
        int((df_raw.isna().sum() > 0).sum()), f"{int(df_raw.duplicated().sum()):,}",
        f"{int(df_raw['EmployeeID'].duplicated().sum()):,}",
        f"{df_raw.memory_usage(deep=True).sum() / 1024**2:.2f}",
    ],
})
display(profile)

# Descriptive statistics for continuous variables
display(df_raw[numeric_cols].describe().T.round(2))

### Column Availability Audit

The analysis plan for this project references a standard HR attrition schema. Before writing any
analysis it is necessary to confirm which of those variables actually exist in **this** dataset,
so that missing variables are reported as unavailable rather than silently invented.

In [ ]:
PLANNED_VARIABLES = [
    "Age", "Attrition", "BusinessTravel", "Department", "DistanceFromHome", "Education",
    "EducationField", "Gender", "JobLevel", "JobRole", "MaritalStatus", "MonthlyIncome",
    "NumCompaniesWorked", "OverTime", "PercentSalaryHike", "PerformanceRating",
    "StockOptionLevel", "TotalWorkingYears", "TrainingTimesLastYear", "YearsAtCompany",
    "YearsInCurrentRole", "YearsSinceLastPromotion", "YearsWithCurrManager",
    "JobSatisfaction", "WorkLifeBalance", "EnvironmentSatisfaction", "JobInvolvement",
]

availability = pd.DataFrame({
    "Planned_Variable": PLANNED_VARIABLES,
    "Status": ["Available" if v in df_raw.columns else "NOT AVAILABLE" for v in PLANNED_VARIABLES],
})
missing_planned = availability.loc[availability.Status == "NOT AVAILABLE", "Planned_Variable"].tolist()

display(availability[availability.Status == "NOT AVAILABLE"])
print(f"Available : {len(PLANNED_VARIABLES) - len(missing_planned)} of {len(PLANNED_VARIABLES)} planned variables")
print(f"Unavailable: {missing_planned}")

extra_cols = [c for c in df_raw.columns if c not in PLANNED_VARIABLES + ID_COLUMNS]
print(f"\nAdditional columns present in this dataset: {extra_cols}")

**Interpretation.**

Two planned variables do not exist in this dataset:

| Variable | Consequence for this analysis |
|---|---|
| `OverTime` | **Section 15 overtime analysis cannot be performed.** No proxy is substituted. Overtime-related questions are marked unavailable and the section covers Business Travel only. |
| `YearsInCurrentRole` | Role-level tenure is unavailable. `YearsWithCurrManager` and `YearsSinceLastPromotion` are used as the available career-progression measures instead. |

The dataset also carries five pre-built **label columns** (`Education_Label`, `JobSatisfaction_Label`,
`EnvironmentSatisfaction_Label`, `WorkLifeBalance_Label`, `PerformanceRating_Label`). These mirror
the numeric codes in readable form and are used for chart labelling — but Section 4 verifies that
each label maps to exactly one code before they are trusted.

---

# 3. Data Quality Assessment

**Objective:** audit the dataset for missing values, duplicates, type problems, invalid values and
logical inconsistencies — and classify every issue found as one of:

- **Data quality issue** → requires a documented correction,
- **Valid extreme value** → unusual but plausible, retained,
- **Potential business exception** → plausible but worth an HR follow-up question.

No record is deleted in this section. Detection and treatment are kept strictly separate.

## 3.1 Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": df_raw.isna().sum(),
    "Missing_Pct": (df_raw.isna().mean() * 100).round(2),
}).sort_values("Missing_Count", ascending=False)

if missing.Missing_Count.sum() == 0:
    print("No missing values detected in any of "
          f"{df_raw.shape[1]} columns across {df_raw.shape[0]:,} rows.")
    print("A missing-value heatmap is therefore omitted: it would be a single uniform block "
          "and would answer no business question.")
else:
    display(missing[missing.Missing_Count > 0])
    plt.figure(figsize=(10, 4))
    sns.heatmap(df_raw.isna(), cbar=False, cmap="viridis")
    plt.title("Missing Value Map")
    plt.tight_layout(); plt.show()

**Interpretation.** The dataset is complete — there are no missing values. This is consistent
with the file being a *cleaned master* export rather than a raw HRIS extract. No imputation is
required, which removes a major source of analytical distortion from everything that follows.

## 3.2 Duplicate Analysis

In [ ]:
full_dups = int(df_raw.duplicated().sum())
id_dups = int(df_raw["EmployeeID"].duplicated().sum())

print(f"Fully duplicated rows : {full_dups:,}  ({full_dups / len(df_raw) * 100:.2f}%)")
print(f"Duplicate EmployeeIDs : {id_dups:,}  ({id_dups / len(df_raw) * 100:.2f}%)")

# Duplicates ignoring the ID: two different IDs with byte-identical attributes would be suspicious
attribute_dups = int(df_raw.drop(columns=ID_COLUMNS).duplicated().sum())
print(f"Rows identical on all attributes except EmployeeID: {attribute_dups:,}")

**Interpretation.** No duplicates of any kind. `EmployeeID` is a genuine unique key
(4,327 distinct IDs for 4,327 rows), so the dataset is confirmed to be at **one row per employee**
grain. Every count in this notebook can therefore be read as an employee headcount.

## 3.3 Data Type Validation

In [ ]:
# Columns stored as float that contain only whole numbers and no nulls are mis-typed.
float_cols = df_raw.select_dtypes(include="float").columns
mistyped = [c for c in float_cols
            if df_raw[c].notna().all() and np.allclose(df_raw[c], df_raw[c].round())]

print("Float columns holding only whole numbers with no missing values (should be integer):")
for c in mistyped:
    print(f"  - {c:<28} range {df_raw[c].min():.0f} to {df_raw[c].max():.0f}")

print(f"\nObject columns (expected for categorical fields): "
      f"{len(df_raw.select_dtypes(include='object').columns)}")

**Interpretation.** Five columns are stored as `float64` but contain only whole numbers and no
nulls — a side effect of the earlier cleaning step, **not** a data problem. They are cast to integer
in Section 4 so that counts and group labels display cleanly (`3.0` → `3`). This changes
presentation only; no value is altered.

## 3.4 Unique Value Analysis

In [ ]:
for col in categorical_cols:
    counts = df_raw[col].value_counts()
    print(f"\n{col}  ({df_raw[col].nunique()} unique)")
    print("  " + " | ".join(f"{k}: {v:,}" for k, v in counts.items()))

In [ ]:
# Verify the pre-built label columns map 1:1 onto their numeric codes.
LABEL_PAIRS = {
    "Education": "Education_Label",
    "JobSatisfaction": "JobSatisfaction_Label",
    "EnvironmentSatisfaction": "EnvironmentSatisfaction_Label",
    "WorkLifeBalance": "WorkLifeBalance_Label",
    "PerformanceRating": "PerformanceRating_Label",
}

mapping_rows = []
for code_col, label_col in LABEL_PAIRS.items():
    codes_per_label = df_raw.groupby(label_col)[code_col].nunique()
    labels_per_code = df_raw.groupby(code_col)[label_col].nunique()
    consistent = (codes_per_label.max() == 1) and (labels_per_code.max() == 1)
    mapping = dict(sorted(df_raw.groupby(code_col)[label_col].first().items()))
    mapping_rows.append({
        "Code_Column": code_col,
        "Label_Column": label_col,
        "One_To_One": "Yes" if consistent else "NO - INCONSISTENT",
        "Mapping": " | ".join(f"{k}={v}" for k, v in mapping.items()),
    })

display(pd.DataFrame(mapping_rows))

**Interpretation.** Every label column maps one-to-one onto its numeric code, with no
contradictions. The label columns are therefore safe to use for chart axes and grouped tables,
while the numeric codes remain available for ordering and correlation work.

Note the scale direction differs between fields and matters for interpretation:
`WorkLifeBalance` runs `1=Bad → 4=Best`, while `JobSatisfaction` runs `1=Low → 4=Very High`.
Both are "higher is better", but they are **not** the same scale and are never pooled.

## 3.5 Invalid Value Detection & HR Logical Constraints

In [ ]:
def run_constraint_checks(data):
    # Evaluate HR business rules and return a tidy findings table (detection only).
    checks = [
        ("Age within working range (18-70)",        (data.Age < 18) | (data.Age > 70)),
        ("MonthlyIncome greater than zero",         data.MonthlyIncome <= 0),
        ("TotalWorkingYears not negative",          data.TotalWorkingYears < 0),
        ("YearsAtCompany not negative",             data.YearsAtCompany < 0),
        ("YearsAtCompany <= TotalWorkingYears",     data.YearsAtCompany > data.TotalWorkingYears),
        ("YearsWithCurrManager <= YearsAtCompany",  data.YearsWithCurrManager > data.YearsAtCompany),
        ("YearsSinceLastPromotion <= YearsAtCompany", data.YearsSinceLastPromotion > data.YearsAtCompany),
        ("TotalWorkingYears plausible vs Age (<= Age-17)", data.TotalWorkingYears > (data.Age - 17)),
        ("Attrition in {Yes, No}",                  ~data.Attrition.isin(["Yes", "No"])),
        ("JobSatisfaction on 1-4 scale",            ~data.JobSatisfaction.isin([1, 2, 3, 4])),
        ("WorkLifeBalance on 1-4 scale",            ~data.WorkLifeBalance.isin([1, 2, 3, 4])),
        ("EnvironmentSatisfaction on 1-4 scale",    ~data.EnvironmentSatisfaction.isin([1, 2, 3, 4])),
        ("JobInvolvement on 1-4 scale",             ~data.JobInvolvement.isin([1, 2, 3, 4])),
        ("Education on 1-5 scale",                  ~data.Education.isin([1, 2, 3, 4, 5])),
        ("JobLevel on 1-5 scale",                   ~data.JobLevel.isin([1, 2, 3, 4, 5])),
        ("PerformanceRating on 1-4 scale",          ~data.PerformanceRating.isin([1, 2, 3, 4])),
        ("PercentSalaryHike within 0-100",          (data.PercentSalaryHike < 0) | (data.PercentSalaryHike > 100)),
        ("TrainingTimesLastYear within 0-52",       (data.TrainingTimesLastYear < 0) | (data.TrainingTimesLastYear > 52)),
        ("NumCompaniesWorked not negative",         data.NumCompaniesWorked < 0),
        ("DistanceFromHome greater than zero",      data.DistanceFromHome <= 0),
    ]
    return pd.DataFrame([
        {"Constraint": name, "Violations": int(mask.sum()),
         "Pct_of_Rows": round(mask.sum() / len(data) * 100, 3)}
        for name, mask in checks
    ])

constraints = run_constraint_checks(df_raw)
display(constraints)
print(f"Constraints passed cleanly: {(constraints.Violations == 0).sum()} of {len(constraints)}")

In [ ]:
# Inspect every record that violated any constraint, rather than reporting counts alone.
violation_mask = (
    (df_raw.YearsAtCompany > df_raw.TotalWorkingYears)
    | (df_raw.YearsWithCurrManager > df_raw.YearsAtCompany)
    | (df_raw.YearsSinceLastPromotion > df_raw.YearsAtCompany)
    | (df_raw.TotalWorkingYears > (df_raw.Age - 17))
)
flagged = df_raw.loc[violation_mask, [
    "EmployeeID", "Age", "Department", "JobRole", "JobLevel",
    "TotalWorkingYears", "YearsAtCompany", "YearsWithCurrManager", "YearsSinceLastPromotion",
]]
print(f"Records violating at least one tenure-logic constraint: {len(flagged)}")
display(flagged)

In [ ]:
# Boundary / zero-value review: unusual but not necessarily invalid
boundary = pd.DataFrame({
    "Observation": [
        "TotalWorkingYears == 0 (no prior work experience)",
        "YearsAtCompany == 0 (joined within the last year)",
        "YearsSinceLastPromotion == 0 (no promotion gap recorded)",
        "NumCompaniesWorked == 0 (this is their first employer)",
        "TrainingTimesLastYear == 0 (no training recorded)",
        "Age <= 20",
        "MonthlyIncome at dataset maximum",
    ],
    "Employees": [
        int((df_raw.TotalWorkingYears == 0).sum()),
        int((df_raw.YearsAtCompany == 0).sum()),
        int((df_raw.YearsSinceLastPromotion == 0).sum()),
        int((df_raw.NumCompaniesWorked == 0).sum()),
        int((df_raw.TrainingTimesLastYear == 0).sum()),
        int((df_raw.Age <= 20).sum()),
        int((df_raw.MonthlyIncome == df_raw.MonthlyIncome.max()).sum()),
    ],
})
boundary["Pct_of_Workforce"] = (boundary.Employees / len(df_raw) * 100).round(2)
display(boundary)

### Data Quality Findings — Classification

| # | Finding | Scale | Classification | Treatment decision |
|---|---|---|---|---|
| 1 | No missing values | 0 cells | Clean | None required |
| 2 | No duplicate rows or IDs | 0 rows | Clean | None required |
| 3 | Five integer fields stored as `float64` | 5 columns | **Data quality issue** (cosmetic) | Cast to `int` — presentation only, values unchanged |
| 4 | `EmployeeID 24`: 20 years at company but 10 total working years | 1 row (0.02%) | **Data quality issue** — logically impossible | Flag with `Data_Quality_Flag`, **retain** the row, exclude it from tenure-ratio analysis only |
| 5 | 32 employees with `TotalWorkingYears = 0` | 0.74% | **Valid** | Genuine new entrants to the workforce |
| 6 | 127 employees with `YearsAtCompany = 0` | 2.94% | **Valid** | Joined within the last year |
| 7 | 1,710 employees with `YearsSinceLastPromotion = 0` | 39.5% | **Potential business exception** | The field records a *gap of zero years*, which conflates "promoted this year" with "never promoted / no promotion recorded". Wording is kept deliberately neutral throughout Section 12 |
| 8 | 161 employees with zero training | 3.7% | **Potential business exception** | Plausible, but worth an HR data-capture question |
| 9 | All rating scales within range; `Attrition` strictly Yes/No | — | Clean | None required |

**On finding 4:** a single impossible row out of 4,327 is a transcription-level error, not a
systemic one. Deleting it would be defensible, but it is retained and flagged instead — the row's
other 30 fields are valid, and dropping employees quietly is exactly the habit this notebook avoids.

## 3.6 Structural Observations That Constrain Interpretation

Two characteristics of this dataset were found during profiling that materially limit what can
honestly be concluded later. They are documented here, up front, rather than discovered by a reader
halfway through the compensation section.

In [ ]:
# Observation A: are job roles confined to sensible departments?
role_dept = pd.crosstab(df_raw.JobRole, df_raw.Department)
display(role_dept)

roles_spanning_all = (role_dept > 0).sum(axis=1).eq(role_dept.shape[1]).sum()
print(f"Job roles appearing in all {role_dept.shape[1]} departments: "
      f"{roles_spanning_all} of {role_dept.shape[0]}")

In [ ]:
# Observation B: does compensation behave the way compensation normally behaves?
structural = pd.DataFrame({
    "Relationship": [
        "MonthlyIncome vs JobLevel",
        "MonthlyIncome vs TotalWorkingYears",
        "MonthlyIncome vs Age",
        "PercentSalaryHike vs PerformanceRating",
        "YearsAtCompany vs YearsWithCurrManager",
    ],
    "Pearson_r": [
        df_raw.MonthlyIncome.corr(df_raw.JobLevel),
        df_raw.MonthlyIncome.corr(df_raw.TotalWorkingYears),
        df_raw.MonthlyIncome.corr(df_raw.Age),
        df_raw.PercentSalaryHike.corr(df_raw.PerformanceRating),
        df_raw.YearsAtCompany.corr(df_raw.YearsWithCurrManager),
    ],
}).round(3)
display(structural)

print("Median MonthlyIncome by JobLevel:")
display(df_raw.groupby("JobLevel").MonthlyIncome.median().to_frame("Median_Income"))

**Interpretation — two structural caveats.**

**A. Job roles are not confined to their natural departments.** Research Scientists appear in Sales,
Sales Executives in R&D, and so on; most roles appear in all three departments. In a real HRIS this
would be impossible. It indicates that `Department` and `JobRole` were assigned independently in the
construction of this dataset.  
→ *Consequence:* department-level and role-level findings are treated as **two separate lenses on
the same workforce**, never combined into a claim such as "Sales Executives in R&D are at risk".

**B. Compensation is effectively independent of job level and experience** (r ≈ 0.05 and ≈ −0.03).
In any real organisation, income rises steeply with seniority; here median income is flat across
levels 1–5. By contrast `PercentSalaryHike` ↔ `PerformanceRating` (r ≈ 0.77) and
`YearsAtCompany` ↔ `YearsWithCurrManager` (r ≈ 0.77) behave exactly as expected.  
→ *Consequence:* the compensation analysis in Section 9 is reported **descriptively**, and this
notebook explicitly does **not** conclude anything about pay equity, pay progression, or
underpayment-driven attrition. The structure required to support such a conclusion is not present
in the data.

Stating these limits is not a weakness of the analysis — it is the part that makes the remaining
conclusions trustworthy.

---

# 4. Data Cleaning & Preprocessing

**Objective:** apply only the corrections justified by Section 3 findings, each one documented and
reversible in intent. The working dataframe `df` is created here; `df_raw` is preserved untouched so
that any before/after comparison can be re-run.

**Transformations applied:**

1. Cast five whole-number `float64` columns to `int` (cosmetic, value-preserving).
2. Strip stray whitespace from categorical text fields.
3. Add a `Data_Quality_Flag` marking the single tenure-inconsistent record — **no row is dropped**.

**Deliberately not done:** no imputation (nothing is missing), no de-duplication (nothing is
duplicated), no outlier removal (see Section 18), no renaming of columns (the existing names are
already clean `PascalCase` and match the SQL layer, so renaming would break SQL↔Python comparison).

In [ ]:
df = df_raw.copy()
before = {"rows": len(df), "cols": df.shape[1],
          "float_cols": int(df.select_dtypes(include="float").shape[1]),
          "memory_mb": df.memory_usage(deep=True).sum() / 1024**2}

# --- Transformation 1: correct integer types ---
INT_CAST_COLS = ["NumCompaniesWorked", "TotalWorkingYears", "EnvironmentSatisfaction",
                 "JobSatisfaction", "WorkLifeBalance"]
values_preserved = all(np.allclose(df[c], df[c].round()) for c in INT_CAST_COLS)
assert values_preserved, "Cast aborted: a column contains fractional values."
df[INT_CAST_COLS] = df[INT_CAST_COLS].astype(int)

# --- Transformation 2: standardise categorical text ---
text_cols = df.select_dtypes(include="object").columns
whitespace_fixed = int(sum((df[c] != df[c].str.strip()).sum() for c in text_cols))
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

# --- Transformation 3: flag (do not delete) the logically inconsistent record ---
inconsistent = (
    (df.YearsAtCompany > df.TotalWorkingYears)
    | (df.YearsWithCurrManager > df.YearsAtCompany)
    | (df.YearsSinceLastPromotion > df.YearsAtCompany)
    | (df.TotalWorkingYears > (df.Age - 17))
)
df["Data_Quality_Flag"] = np.where(inconsistent, "Tenure_Inconsistent", "OK")

after = {"rows": len(df), "cols": df.shape[1],
         "float_cols": int(df.select_dtypes(include="float").shape[1]),
         "memory_mb": df.memory_usage(deep=True).sum() / 1024**2}

summary = pd.DataFrame({
    "Check": ["Rows", "Columns", "Float columns", "Memory (MB)", "Whitespace fixes",
              "Rows flagged (retained)", "Rows deleted"],
    "Before": [f"{before['rows']:,}", before["cols"], before["float_cols"],
               f"{before['memory_mb']:.2f}", "-", "-", "-"],
    "After": [f"{after['rows']:,}", after["cols"], after["float_cols"],
              f"{after['memory_mb']:.2f}", whitespace_fixed,
              int((df.Data_Quality_Flag != "OK").sum()), before["rows"] - after["rows"]],
})
display(summary)

In [ ]:
# Confirm the cleaning did not alter any analytical value
assert len(df) == len(df_raw), "Row count changed unexpectedly"
assert df.MonthlyIncome.sum() == df_raw.MonthlyIncome.sum(), "Income total changed"
assert df.Attrition.value_counts().to_dict() == df_raw.Attrition.value_counts().to_dict()
print("Integrity verified: row count, income total and attrition distribution are unchanged.")
print(f"Records retained with a quality flag: {(df.Data_Quality_Flag != 'OK').sum()}")

**Interpretation.** Cleaning was minimal by design — the input file had genuinely already been
cleaned, and the audit confirmed it independently rather than taking it on trust. The only
substantive change is a flag column; **zero rows were removed**, and the assertions above prove that
headcount, total payroll and the attrition distribution are byte-identical to the source file.

---

# 5. Feature Engineering

**Objective:** derive analytical features that make group comparison possible. Every band boundary
below is justified either by the **actual distribution** of the variable or by **alignment with the
existing SQL logic** — no bin is chosen arbitrarily.

In [ ]:
# Inspect the distributions that will drive the binning decisions
binning_basis = df[["Age", "TotalWorkingYears", "YearsAtCompany",
                    "MonthlyIncome", "PercentSalaryHike"]].describe(
    percentiles=[.1, .25, .5, .75, .9]).T.round(1)
display(binning_basis)

### Binning Rationale

| Feature | Boundaries | Justification |
|---|---|---|
| `Attrition_Flag` | Yes→1, No→0 | Enables `.mean()` to return the attrition **rate** directly, which is the core operation of this notebook |
| `Age_Group` | 18–24, 25–34, 35–44, 45–54, 55+ | **Matches SQL Query 17 exactly**, so Python and SQL age analysis are directly comparable. Also standard HR career-stage bands |
| `Experience_Group` | 0–2, 3–5, 6–10, 11–15, 16+ | Career-stage bands on `TotalWorkingYears`; median is 10 years, so the split falls naturally around the centre of the distribution |
| `Tenure_Group` | 0–2, 3–5, 6–10, 11–20, 20+ | Based on the actual `YearsAtCompany` distribution (median 5, P75 = 9, max 40). A single 11+ band would hide the long-tenure tail, so it is split at 20 |
| `Income_Band` | Quartiles (Q1–Q4) | Income is right-skewed (mean ≫ median), so fixed-width bins would leave near-empty top bands. Quartiles guarantee ~1,082 employees per band, which keeps every attrition rate statistically comparable |
| `Salary_Hike_Band` | 11–13, 14–16, 17–19, 20–25 | `PercentSalaryHike` ranges 11–25 with only 15 distinct values; 3-point bands map onto the natural clustering of the distribution |

Quartile-based income banding is chosen deliberately over fixed currency bands: comparing attrition
rates between a band of 1,500 employees and a band of 40 employees would produce unstable,
misleading rates.

In [ ]:
# ---------- 1. Attrition flag ----------
df["Attrition_Flag"] = (df.Attrition == "Yes").astype(int)

# ---------- 2. Age group (aligned with SQL Query 17) ----------
df["Age_Group"] = pd.cut(df.Age, bins=[17, 24, 34, 44, 54, np.inf],
                         labels=["18-24", "25-34", "35-44", "45-54", "55+"])

# ---------- 3. Experience group (total career experience) ----------
df["Experience_Group"] = pd.cut(df.TotalWorkingYears, bins=[-1, 2, 5, 10, 15, np.inf],
                                labels=["0-2", "3-5", "6-10", "11-15", "16+"])

# ---------- 4. Tenure group (time at this company) ----------
df["Tenure_Group"] = pd.cut(df.YearsAtCompany, bins=[-1, 2, 5, 10, 20, np.inf],
                            labels=["0-2", "3-5", "6-10", "11-20", "20+"])

# ---------- 5. Income band (quartiles of the observed distribution) ----------
df["Income_Band"] = pd.qcut(df.MonthlyIncome, q=4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])

# ---------- 6. Salary hike band ----------
df["Salary_Hike_Band"] = pd.cut(df.PercentSalaryHike, bins=[10, 13, 16, 19, np.inf],
                                labels=["11-13%", "14-16%", "17-19%", "20%+"])

# ---------- 7. Promotion status (neutral wording - see Section 3 finding 7) ----------
df["Promotion_Status"] = np.where(df.YearsSinceLastPromotion == 0,
                                  "No promotion gap recorded", "1+ years since promotion")

new_features = ["Attrition_Flag", "Age_Group", "Experience_Group", "Tenure_Group",
                "Income_Band", "Salary_Hike_Band", "Promotion_Status"]
print(f"Created {len(new_features)} features. Dataset now {df.shape[0]:,} x {df.shape[1]}")

In [ ]:
# Verify every band is populated and no employee was lost to a NaN bin
for feature in ["Age_Group", "Experience_Group", "Tenure_Group", "Income_Band",
                "Salary_Hike_Band", "Promotion_Status"]:
    counts = df[feature].value_counts().sort_index()
    share = (counts / len(df) * 100).round(1)
    combined = " | ".join(f"{idx}: {c:,} ({s}%)" for idx, c, s in zip(counts.index, counts, share))
    print(f"{feature:<20} -> {combined}")
    assert df[feature].isna().sum() == 0, f"{feature} produced unassigned values"

print("\nIncome band boundaries (actual currency values):")
display(df.groupby("Income_Band", observed=True).MonthlyIncome
          .agg(Employees="size", Min="min", Max="max", Median="median"))

,Employees,Min,Max,Median
Income_Band,,,,
Q1 (Lowest),1084,10090,29260,"23,790.00"
Q2,1083,29290,49360,"40,510.00"
Q3,1078,49410,83800,"60,910.00"
Q4 (Highest),1082,83810,199990,"131,160.00"


**Interpretation.** All seven features are fully populated with no unassigned records. The
distributions are workable for comparison: the smallest band is `20%+` salary hike and the `55+`
age group, both still large enough (200+ employees) for a stable rate. The income quartile
boundaries land at roughly 29K / 49K / 84K, confirming the right-skew that motivated quantile
binning over fixed-width bins.

---

# 6. Workforce Overview

**Objective:** establish who the workforce actually is before asking who leaves it. Every attrition
rate later in the notebook has to be read against the size of the group it describes, so the
composition analysis comes first.

The reusable helper functions defined here are used throughout the rest of the notebook to avoid
duplicated plotting and aggregation code.

In [ ]:
def attrition_summary(data, group_col, sort_by="Attrition_Rate_%", min_group_size=0):
    # Return employee count, attrition count and attrition RATE for each level of group_col.
    # min_group_size suppresses groups too small to support a stable rate.
    summary = (data.groupby(group_col, observed=True)
                   .agg(Employees=("Attrition_Flag", "size"),
                        Attrition_Count=("Attrition_Flag", "sum"))
                   .assign(Attrition_Rate_=lambda d: (d.Attrition_Count / d.Employees * 100).round(2))
                   .rename(columns={"Attrition_Rate_": "Attrition_Rate_%"}))
    summary = summary[summary.Employees >= min_group_size]
    return summary.sort_values(sort_by, ascending=False)


def plot_attrition_rate(summary, title, horizontal=True, benchmark=None, figsize=(10, 5)):
    # Plot attrition RATE (never raw count) with an optional company-average reference line.
    data = summary.sort_values("Attrition_Rate_%")
    fig, ax = plt.subplots(figsize=figsize)
    colors = [ACCENT if v >= (benchmark or 0) else NEUTRAL for v in data["Attrition_Rate_%"]]

    if horizontal:
        ax.barh(data.index.astype(str), data["Attrition_Rate_%"], color=colors)
        ax.set_xlabel("Attrition Rate (%)"); ax.set_ylabel("")
        for y, (rate, n) in enumerate(zip(data["Attrition_Rate_%"], data.Employees)):
            ax.text(rate + 0.3, y, f"{rate:.1f}%  (n={n:,})", va="center", fontsize=9)
        ax.set_xlim(0, data["Attrition_Rate_%"].max() * 1.30)
        if benchmark is not None:
            ax.axvline(benchmark, color="black", ls="--", lw=1,
                       label=f"Company average {benchmark:.1f}%")
            ax.legend(loc="lower right", fontsize=9)
    else:
        ax.bar(data.index.astype(str), data["Attrition_Rate_%"], color=colors)
        ax.set_ylabel("Attrition Rate (%)"); ax.set_xlabel("")
        for x, (rate, n) in enumerate(zip(data["Attrition_Rate_%"], data.Employees)):
            ax.text(x, rate + 0.4, f"{rate:.1f}%\n(n={n:,})", ha="center", fontsize=9)
        ax.set_ylim(0, data["Attrition_Rate_%"].max() * 1.25)
        if benchmark is not None:
            ax.axhline(benchmark, color="black", ls="--", lw=1,
                       label=f"Company average {benchmark:.1f}%")
            ax.legend(fontsize=9)

    ax.set_title(title)
    plt.tight_layout(); plt.show()


def plot_composition(data, col, title, horizontal=True, figsize=(10, 4.5), top_n=None):
    # Plot headcount composition for a categorical variable.
    counts = data[col].value_counts()
    if top_n:
        counts = counts.head(top_n)
    counts = counts.sort_values()
    fig, ax = plt.subplots(figsize=figsize)
    total = len(data)
    if horizontal:
        ax.barh(counts.index.astype(str), counts.values, color=NEUTRAL)
        ax.set_xlabel("Number of Employees")
        for y, v in enumerate(counts.values):
            ax.text(v + total * 0.004, y, f"{v:,} ({v/total*100:.1f}%)", va="center", fontsize=9)
        ax.set_xlim(0, counts.max() * 1.22)
    else:
        ax.bar(counts.index.astype(str), counts.values, color=NEUTRAL)
        ax.set_ylabel("Number of Employees")
        for x, v in enumerate(counts.values):
            ax.text(x, v + total * 0.005, f"{v:,}\n({v/total*100:.1f}%)", ha="center", fontsize=9)
        ax.set_ylim(0, counts.max() * 1.18)
    ax.set_title(title)
    plt.tight_layout(); plt.show()


def thousands_formatter(ax, axis="y"):
    # Format a currency axis in thousands for readability.
    fmt = mticker.FuncFormatter(lambda v, _: f"{v/1000:,.0f}K")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)


print("Helper functions ready: attrition_summary, plot_attrition_rate, plot_composition, thousands_formatter")

## 6.1 Organisational Structure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

dept = df.Department.value_counts().sort_values()
axes[0].barh(dept.index, dept.values, color=NEUTRAL)
axes[0].set_title("Employees by Department"); axes[0].set_xlabel("Number of Employees")
for y, v in enumerate(dept.values):
    axes[0].text(v + 40, y, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontsize=9)
axes[0].set_xlim(0, dept.max() * 1.3)

level = df.JobLevel.value_counts().sort_index()
axes[1].bar(level.index.astype(str), level.values, color=NEUTRAL)
axes[1].set_title("Employees by Job Level"); axes[1].set_xlabel("Job Level (1 = Junior)")
axes[1].set_ylabel("Number of Employees")
for x, v in enumerate(level.values):
    axes[1].text(x, v + 25, f"{v:,}", ha="center", fontsize=9)
axes[1].set_ylim(0, level.max() * 1.15)

plt.tight_layout(); plt.show()

In [ ]:
plot_composition(df, "JobRole", "Employees by Job Role", figsize=(10, 5))

**Interpretation.** The workforce is concentrated in Research & Development (2,824 employees,
65%), with Sales at 30% and Human Resources a small function at just 188 employees (4.3%). That HR
headcount matters for later reading: a handful of exits there moves its attrition rate several
percentage points, whereas the same number of exits in R&D is statistically invisible.

Job level is pyramid-shaped as expected — levels 1 and 2 hold 73% of employees and level 5 only 204.
Sales Executive, Research Scientist and Laboratory Technician together account for over half the
workforce.

## 6.2 Workforce Composition Hierarchy

A single interactive chart is used here, where the department → role hierarchy genuinely benefits
from drill-down. (Renders interactively in Jupyter; static viewers should refer to the tables above.)

In [ ]:
hierarchy = (df.groupby(["Department", "JobRole"], observed=True)
               .size().reset_index(name="Employees"))

fig = px.sunburst(hierarchy, path=["Department", "JobRole"], values="Employees",
                  color="Employees", color_continuous_scale="Blues",
                  title="Workforce Composition: Department to Job Role")
fig.update_layout(width=720, height=520, margin=dict(t=60, l=0, r=0, b=0))
fig.show()

# Underlying figures, so the composition is readable even where interactive output does not render
display(hierarchy.pivot(index="JobRole", columns="Department", values="Employees")
                 .fillna(0).astype(int))

## 6.3 Demographic Composition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, (col, title) in zip(axes.flat, [
    ("Gender", "Employees by Gender"),
    ("MaritalStatus", "Employees by Marital Status"),
    ("Education_Label", "Employees by Education Level"),
    ("BusinessTravel", "Employees by Business Travel Frequency"),
]):
    counts = df[col].value_counts().sort_values()
    ax.barh(counts.index.astype(str), counts.values, color=NEUTRAL)
    ax.set_title(title); ax.set_xlabel("Number of Employees")
    for y, v in enumerate(counts.values):
        ax.text(v + len(df) * 0.006, y, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontsize=9)
    ax.set_xlim(0, counts.max() * 1.32)

plt.tight_layout(); plt.show()

In [ ]:
plot_composition(df, "EducationField", "Employees by Education Field", figsize=(10, 4.5))

**Interpretation.** The workforce is 60% male / 40% female and predominantly married (46%).
Education is concentrated at Bachelor (39%) and Master (27%) level, with only 141 Doctorates.
Educational background is heavily science-weighted — Life Sciences and Medical together account for
73% of employees, consistent with an R&D-dominant organisation. Travel is the norm but not intensive:
71% travel rarely, 19% travel frequently, and only 10% do not travel at all.

## 6.4 Continuous Distributions: Age and Tenure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].hist(df.Age, bins=25, color=NEUTRAL, edgecolor="white")
axes[0].axvline(df.Age.median(), color=ACCENT, ls="--", label=f"Median {df.Age.median():.0f}")
axes[0].set_title("Age Distribution"); axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Number of Employees"); axes[0].legend(fontsize=9)

axes[1].hist(df.YearsAtCompany, bins=30, color=NEUTRAL, edgecolor="white")
axes[1].axvline(df.YearsAtCompany.median(), color=ACCENT, ls="--",
                label=f"Median {df.YearsAtCompany.median():.0f}")
axes[1].set_title("Years at Company Distribution"); axes[1].set_xlabel("Years at Company")
axes[1].set_ylabel("Number of Employees"); axes[1].legend(fontsize=9)

axes[2].hist(df.TotalWorkingYears, bins=30, color=NEUTRAL, edgecolor="white")
axes[2].axvline(df.TotalWorkingYears.median(), color=ACCENT, ls="--",
                label=f"Median {df.TotalWorkingYears.median():.0f}")
axes[2].set_title("Total Working Years Distribution"); axes[2].set_xlabel("Total Working Years")
axes[2].set_ylabel("Number of Employees"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(df[["Age", "YearsAtCompany", "TotalWorkingYears"]]
      .agg(["mean", "median", "std", "min", "max"]).round(1))

**Interpretation.** Age is near-symmetric around a median of 36 (range 18–60) — a mature but
not ageing workforce. Company tenure is strongly right-skewed: median 5 years, but with a long tail
out to 40 years. Roughly a quarter of employees have been with the company 2 years or less, which is
the segment most exposed to early-tenure attrition risk examined in Section 8.

---

# 7. Core Attrition Analysis

**Objective:** this is the primary analytical section. It quantifies overall attrition and then
decomposes it across every meaningful employee segment.

**The central methodological rule of this section:** attrition **count** and attrition **rate** are
never confused. Count tells you where the volume of exits sits; rate tells you where the *risk*
sits. Comparing raw counts across groups of different sizes — R&D's 2,824 employees against HR's
188 — would systematically point HR at the largest department rather than the most affected one.

## 7.1 Headline Attrition Metrics

In [ ]:
total_employees = len(df)
left = int(df.Attrition_Flag.sum())
stayed = total_employees - left
attrition_rate = df.Attrition_Flag.mean() * 100

headline = pd.DataFrame({
    "Metric": ["Total Employees", "Employees Who Left", "Employees Who Stayed",
               "Overall Attrition Rate", "Retention Rate"],
    "Value": [f"{total_employees:,}", f"{left:,}", f"{stayed:,}",
              f"{attrition_rate:.2f}%", f"{100 - attrition_rate:.2f}%"],
})
display(headline)

COMPANY_ATTRITION_RATE = attrition_rate  # benchmark used on every chart in this section
print(f"Benchmark for all segment comparisons: {COMPANY_ATTRITION_RATE:.2f}%")

**Interpretation.** 701 of 4,327 employees have left — an overall attrition rate of **16.20%**.
For context, that sits in the range typically considered elevated-but-not-critical for a
professional-services or R&D workforce. The organisational question is not whether 16% is good or
bad in the abstract, but **whether it is evenly distributed**. The rest of this section shows it is
not: segment rates range from roughly 6% to over 40%.

## 7.2 Attrition by Department and Job Role

In [ ]:
dept_attrition = attrition_summary(df, "Department")
display(dept_attrition)

In [ ]:
# Count vs rate side by side - the distinction that drives correct HR targeting
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ordered = dept_attrition.sort_values("Attrition_Count")

axes[0].barh(ordered.index, ordered.Attrition_Count, color=NEUTRAL)
axes[0].set_title("Attrition COUNT by Department\n(where the volume of exits is)")
axes[0].set_xlabel("Employees Who Left")
for y, v in enumerate(ordered.Attrition_Count):
    axes[0].text(v + 6, y, f"{v:,}", va="center", fontsize=9)
axes[0].set_xlim(0, ordered.Attrition_Count.max() * 1.18)

ordered_rate = dept_attrition.sort_values("Attrition_Rate_%")
axes[1].barh(ordered_rate.index, ordered_rate["Attrition_Rate_%"], color=ACCENT)
axes[1].axvline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
axes[1].set_title("Attrition RATE by Department\n(where the risk is)")
axes[1].set_xlabel("Attrition Rate (%)"); axes[1].legend(fontsize=8, loc="lower right")
for y, v in enumerate(ordered_rate["Attrition_Rate_%"]):
    axes[1].text(v + 0.4, y, f"{v:.1f}%", va="center", fontsize=9)
axes[1].set_xlim(0, ordered_rate["Attrition_Rate_%"].max() * 1.28)

plt.tight_layout(); plt.show()

**Interpretation — the count/rate distinction in practice.** R&D loses the most people in
absolute terms (447 exits) simply because it is the largest department; its rate (15.83%) is actually
slightly *below* the company average. Human Resources loses only 56 people, but at a rate of
**29.79% — nearly double the company average and the highest of the three departments**.

An HR programme prioritised on exit volume would target R&D. A programme prioritised on risk would
target the HR function. The rate-based reading is the correct one for retention strategy; the
count-based reading remains useful for workforce-planning and replacement-cost budgeting.

*Caveat:* with 188 employees, HR's rate is based on a small base and will be more volatile
year-to-year than R&D's. It should be confirmed against a second period before major investment.

In [ ]:
role_attrition = attrition_summary(df, "JobRole")
display(role_attrition)
plot_attrition_rate(role_attrition, "Attrition Rate by Job Role",
                    benchmark=COMPANY_ATTRITION_RATE, figsize=(10, 5))

**Interpretation.** Research Director shows the highest role-level attrition at **23.95%**
(57 of 238), followed by Research Scientist at 18.42%. Manufacturing Director is the most stable role
at 11.21%. The spread across roles (11%–24%) is narrower than the spread across age or tenure groups
seen below, which suggests role is a weaker differentiator of attrition in this dataset than
career-stage variables are — a point the chi-square tests in Section 20 quantify.

## 7.3 Attrition by Demographics

In [ ]:
def small_multiple_attrition(dims, ncols, figsize, suptitle):
    # One combined figure of small attrition-rate panels, instead of a full-page
    # chart per dimension - keeps every comparison but avoids repeating the same
    # chart type six times in a row.
    nrows = -(-len(dims) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    summaries = {}

    for ax, dim in zip(axes, dims):
        summary = attrition_summary(df, dim).sort_values("Attrition_Rate_%")
        summaries[dim] = summary
        colors = [ACCENT if v >= COMPANY_ATTRITION_RATE else NEUTRAL
                  for v in summary["Attrition_Rate_%"]]
        ax.barh(summary.index.astype(str), summary["Attrition_Rate_%"], color=colors)
        ax.axvline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1)
        ax.set_title(dim.replace("_", " "), fontsize=11)
        ax.set_xlabel("Attrition Rate (%)")
        for y, v in enumerate(summary["Attrition_Rate_%"]):
            ax.text(v + 0.5, y, f"{v:.1f}%", va="center", fontsize=8.5)
        ax.set_xlim(0, summary["Attrition_Rate_%"].max() * 1.30)

    for ax in axes[len(dims):]:
        ax.axis("off")

    fig.suptitle(f"{suptitle}  (dashed line = company average {COMPANY_ATTRITION_RATE:.1f}%)",
                 y=1.02, fontsize=12.5, fontweight="bold")
    plt.tight_layout(); plt.show()
    return summaries


demographic_summaries = small_multiple_attrition(
    dims=["Gender", "MaritalStatus", "Age_Group", "Education_Label"],
    ncols=2, figsize=(13, 8.5), suptitle="Attrition Rate by Demographic Segment")

for dim, summary in demographic_summaries.items():
    print(f"\n{dim}:")
    display(summary)

**Interpretation.**

- **Gender** shows almost no difference: 16.76% male vs 15.37% female. A 1.4-point gap on samples of
  this size is unlikely to be meaningful — Section 20 confirms it is not statistically significant.
- **Marital status** shows one of the sharpest divides in the dataset: Single employees leave at
  **25.52%**, Married at 12.72% and Divorced at 9.91%. Single employees are leaving at roughly
  **2.6× the rate of divorced employees**. Marital status is almost certainly acting as a proxy
  here — it correlates with age, tenure and life stage — rather than being a driver in itself.
- **Age group** is the strongest single demographic signal: **39.24% of employees aged 18–24 have
  left**, against 10.11% of those aged 35–44. Attrition falls monotonically until age 55+.
- **Education level** shows little meaningful variation (14.9%–18.7%), and the ordering is not
  monotonic, so no education gradient should be read into it.

In [ ]:
field_attrition = attrition_summary(df, "EducationField")
display(field_attrition)
plot_attrition_rate(field_attrition, "Attrition Rate by Education Field",
                    benchmark=COMPANY_ATTRITION_RATE, figsize=(10, 4))

**Interpretation.** Employees with a Human Resources educational background show by far the
highest attrition at **40.74%** — but this group contains only 81 employees, so the rate rests on 33
exits. It is flagged as notable rather than conclusive. All remaining fields cluster tightly between
11.6% and 16.7%.

## 7.4 Attrition by Job Level, Travel, Income, Experience and Tenure

In [ ]:
structural_summaries = small_multiple_attrition(
    dims=["JobLevel", "BusinessTravel", "Income_Band", "Experience_Group", "Tenure_Group"],
    ncols=3, figsize=(15, 8.5),
    suptitle="Attrition Rate by Job Level, Travel, Income, Experience & Tenure")

for dim, summary in structural_summaries.items():
    print(f"\n{dim}:")
    display(summary)

**Interpretation.**

- **Job level** is remarkably flat (12.75%–17.60%) with no clear seniority gradient. Given the
  Section 3 finding that income is also flat across job levels, `JobLevel` appears to carry limited
  signal in this dataset.
- **Business travel** shows a clean monotonic pattern: Non-Travel 8.14% → Travel_Rarely 15.08% →
  Travel_Frequently **24.79%**. Frequent travellers leave at roughly **3× the rate** of
  non-travellers. This is one of the strongest and most internally consistent patterns in the data.
- **Income band** is nearly flat (13.31%–18.55%) and — importantly — **not monotonic**: the lowest
  and third quartiles both sit near 18%. This is consistent with the structural finding in Section 3
  that income in this dataset does not behave like real compensation data. **No pay-driven attrition
  conclusion is drawn.**
- **Experience group** falls steeply with career experience: 0–2 years at the top, 16+ years lowest.
- **Tenure group** is the single sharpest split found: **30.23% attrition in the 0–2 year band**
  versus 6.79% at 11–20 years — a **4.5× difference**. Early tenure is where this organisation loses
  people.

## 7.5 Consolidated Segment Comparison

A single table ranking every segment analysed above, so HR can see the full risk landscape at once.

In [ ]:
segments = ["Department", "JobRole", "Gender", "Age_Group", "Education_Label",
            "EducationField", "MaritalStatus", "JobLevel", "BusinessTravel",
            "Income_Band", "Experience_Group", "Tenure_Group", "JobSatisfaction_Label",
            "WorkLifeBalance_Label"]

rows = []
for dim in segments:
    s = attrition_summary(df, dim)
    rows.append({
        "Dimension": dim,
        "Levels": len(s),
        "Highest_Risk_Segment": f"{s.index[0]} ({s['Attrition_Rate_%'].iloc[0]:.1f}%, n={s.Employees.iloc[0]:,})",
        "Lowest_Risk_Segment": f"{s.index[-1]} ({s['Attrition_Rate_%'].iloc[-1]:.1f}%, n={s.Employees.iloc[-1]:,})",
        "Rate_Spread_pp": round(s["Attrition_Rate_%"].max() - s["Attrition_Rate_%"].min(), 1),
    })

segment_ranking = pd.DataFrame(rows).sort_values("Rate_Spread_pp", ascending=False)
display(segment_ranking.reset_index(drop=True))

**Interpretation.** Ranking dimensions by the *spread* between their best and worst segment
gives a quick read on which variables actually differentiate attrition. Career-experience band
(35.0 points), education field (29.1) and age group (29.1) produce the widest spreads, followed by
company tenure (23.4). At the other end, gender (1.4), education level (3.8), job level (4.9) and
income band (5.2) barely differentiate at all. Spread alone does not prove significance — small segments can produce large spreads by
chance — which is why Section 20 tests the leading candidates formally.

---

# 8. Compensation Analysis

**Objective:** describe how compensation is distributed across the organisation and examine its
relationship with attrition.

**Important caveat carried forward from Section 3:** in this dataset `MonthlyIncome` is effectively
uncorrelated with `JobLevel` (r ≈ 0.05) and with `TotalWorkingYears` (r ≈ −0.03). Real compensation
data does not behave this way. This section therefore reports **what the data shows** and explicitly
**declines to draw pay-equity or pay-driven-attrition conclusions**, because the data structure
cannot support them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].hist(df.MonthlyIncome, bins=40, color=NEUTRAL, edgecolor="white")
axes[0].axvline(df.MonthlyIncome.median(), color=ACCENT, ls="--",
                label=f"Median {df.MonthlyIncome.median()/1000:,.0f}K")
axes[0].axvline(df.MonthlyIncome.mean(), color=POSITIVE, ls=":",
                label=f"Mean {df.MonthlyIncome.mean()/1000:,.0f}K")
axes[0].set_title("Monthly Income Distribution"); axes[0].set_xlabel("Monthly Income")
axes[0].set_ylabel("Number of Employees"); axes[0].legend(fontsize=9)
thousands_formatter(axes[0], "x")

axes[1].hist(df.PercentSalaryHike, bins=15, color=NEUTRAL, edgecolor="white")
axes[1].axvline(df.PercentSalaryHike.median(), color=ACCENT, ls="--",
                label=f"Median {df.PercentSalaryHike.median():.0f}%")
axes[1].set_title("Percent Salary Hike Distribution"); axes[1].set_xlabel("Salary Hike (%)")
axes[1].set_ylabel("Number of Employees"); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(df[["MonthlyIncome", "PercentSalaryHike"]].describe().T.round(1))

**Interpretation.** Income is strongly right-skewed — mean (65,029) sits well above median
(49,360), the classic signature of a compensation distribution with a long upper tail. Salary hike is
bounded between 11% and 25% with a median of 14%, and is heavily concentrated in the 11–14% range;
the higher hikes form a distinct secondary cluster, explained below by performance rating.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

order_dept = df.groupby("Department").MonthlyIncome.median().sort_values().index
sns.boxplot(data=df, y="Department", x="MonthlyIncome", order=order_dept,
            color=NEUTRAL, ax=axes[0], fliersize=2)
axes[0].set_title("Monthly Income by Department"); axes[0].set_xlabel("Monthly Income")
axes[0].set_ylabel(""); thousands_formatter(axes[0], "x")

order_role = df.groupby("JobRole").MonthlyIncome.median().sort_values().index
sns.boxplot(data=df, y="JobRole", x="MonthlyIncome", order=order_role,
            color=NEUTRAL, ax=axes[1], fliersize=2)
axes[1].set_title("Monthly Income by Job Role"); axes[1].set_xlabel("Monthly Income")
axes[1].set_ylabel(""); thousands_formatter(axes[1], "x")

plt.tight_layout(); plt.show()

comp_table = (df.groupby("Department")
                .agg(Employees=("MonthlyIncome", "size"),
                     Mean_Income=("MonthlyIncome", "mean"),
                     Median_Income=("MonthlyIncome", "median"),
                     Total_Salary_Spend=("MonthlyIncome", "sum"))
                .sort_values("Total_Salary_Spend", ascending=False).round(0))
display(comp_table)

**Interpretation.** Median income differs very little across departments and roles — the
boxplots overlap almost entirely, and every group spans nearly the full income range. Mean income by
department ranges only from 57,917 (HR) to 67,240 (R&D), a spread of under 16%.

Total salary spend, by contrast, is driven almost entirely by headcount: R&D accounts for roughly
two-thirds of payroll because it employs two-thirds of the workforce. This is the correct metric for
budget planning, and it is distinct from average pay.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

sns.boxplot(data=df, x="JobLevel", y="MonthlyIncome", color=NEUTRAL, ax=axes[0], fliersize=2)
axes[0].set_title("Monthly Income by Job Level\n(expected to rise with seniority - it does not)")
axes[0].set_xlabel("Job Level"); axes[0].set_ylabel("Monthly Income")
thousands_formatter(axes[0], "y")

sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition",
            palette=ATTRITION_PALETTE, legend=False, ax=axes[1], fliersize=2)
axes[1].set_title("Monthly Income by Attrition Status")
axes[1].set_xlabel("Left the Company"); axes[1].set_ylabel("Monthly Income")
thousands_formatter(axes[1], "y")

plt.tight_layout(); plt.show()

income_by_attrition = (df.groupby("Attrition")
                         .MonthlyIncome.agg(["size", "mean", "median", "std"]).round(0))
display(income_by_attrition)

**Interpretation — and an explicit non-conclusion.** Median income is 49,000 for leavers and
49,410 for stayers — a difference of 0.8%. Mean income differs more (61,432 vs 65,725), but that gap
is driven by the skewed upper tail rather than by the typical employee.

Section 20 tests this formally and finds that the **median difference is not statistically
significant** (Mann-Whitney U, p ≈ 0.07), even though a t-test on the means is (p ≈ 0.02) — a
textbook example of why the test must match the distribution shape.

Combined with the flat income-vs-job-level relationship, the defensible conclusion is:
**this dataset provides no evidence that compensation level differentiates leavers from stayers.**
Anyone wishing to investigate pay-driven attrition would need salary benchmarked against market rate
and role band, neither of which is present here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

hike_dept = df.groupby("Department").PercentSalaryHike.mean().sort_values()
axes[0].barh(hike_dept.index, hike_dept.values, color=NEUTRAL)
axes[0].set_title("Average Salary Hike by Department"); axes[0].set_xlabel("Average Salary Hike (%)")
for y, v in enumerate(hike_dept.values):
    axes[0].text(v + 0.1, y, f"{v:.2f}%", va="center", fontsize=9)
axes[0].set_xlim(0, hike_dept.max() * 1.18)

hike_band = attrition_summary(df, "Salary_Hike_Band").sort_index()
axes[1].bar(hike_band.index.astype(str), hike_band["Attrition_Rate_%"], color=ACCENT)
axes[1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
axes[1].set_title("Attrition Rate by Salary Hike Band"); axes[1].set_xlabel("Salary Hike Band")
axes[1].set_ylabel("Attrition Rate (%)"); axes[1].legend(fontsize=9)
for x, (v, n) in enumerate(zip(hike_band["Attrition_Rate_%"], hike_band.Employees)):
    axes[1].text(x, v + 0.3, f"{v:.1f}%\n(n={n:,})", ha="center", fontsize=9)
axes[1].set_ylim(0, hike_band["Attrition_Rate_%"].max() * 1.28)

plt.tight_layout(); plt.show()
display(hike_band)

In [ ]:
# Salary vs experience - tested visually rather than assumed
fig, ax = plt.subplots(figsize=(10, 4.6))
sample = df.sample(min(1500, len(df)), random_state=RANDOM_STATE)
ax.scatter(sample.TotalWorkingYears, sample.MonthlyIncome,
           c=sample.Attrition_Flag.map({0: NEUTRAL, 1: ACCENT}), alpha=0.45, s=18)
ax.set_title("Monthly Income vs Total Working Years (1,500-employee sample)")
ax.set_xlabel("Total Working Years"); ax.set_ylabel("Monthly Income")
thousands_formatter(ax, "y")
handles = [plt.Line2D([], [], marker="o", ls="", color=NEUTRAL, label="Stayed"),
           plt.Line2D([], [], marker="o", ls="", color=ACCENT, label="Left")]
ax.legend(handles=handles, fontsize=9)
plt.tight_layout(); plt.show()

print(f"Pearson correlation, income vs experience: {df.MonthlyIncome.corr(df.TotalWorkingYears):.3f}")
print(f"Spearman correlation, income vs experience: "
      f"{df.MonthlyIncome.corr(df.TotalWorkingYears, method='spearman'):.3f}")

**Interpretation.** Average salary hike is essentially identical across departments
(14.98%–15.26%), indicating a uniform hike policy rather than departmental differentiation.
Attrition by hike band is flat and non-monotonic, offering no support for the intuition that smaller
raises drive exits.

The income-vs-experience scatter confirms the structural caveat visually: the cloud is a
featureless rectangle. There is no upward slope, no fan shape, no ceiling effect — nothing that a
real payroll would show. Correlation is essentially zero on both Pearson and Spearman measures.

---

# 9. Job Satisfaction & Work-Life Balance

**Objective:** examine the two headline employee-experience measures individually and in
combination against attrition.

**A note on interpretation before the analysis.** Everything in this section is an **association**.
Even a clean, monotonic relationship between low satisfaction and high attrition cannot establish
that dissatisfaction *causes* exits — satisfaction is measured at a point in time, plausibly *after*
an employee has already decided to leave, and both may be driven by a third factor such as a
difficult manager or an unwanted role change. Establishing causation would require longitudinal
measurement or an intervention design, neither of which exists here.

In [ ]:
SAT_ORDER = ["Low", "Medium", "High", "Very High"]
WLB_ORDER = ["Bad", "Good", "Better", "Best"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

sat_counts = df.JobSatisfaction_Label.value_counts().reindex(SAT_ORDER)
axes[0, 0].bar(SAT_ORDER, sat_counts.values, color=NEUTRAL)
axes[0, 0].set_title("Job Satisfaction Distribution"); axes[0, 0].set_ylabel("Number of Employees")
for x, v in enumerate(sat_counts.values):
    axes[0, 0].text(x, v + 20, f"{v:,}", ha="center", fontsize=9)
axes[0, 0].set_ylim(0, sat_counts.max() * 1.15)

sat_rate = attrition_summary(df, "JobSatisfaction_Label").reindex(SAT_ORDER)
axes[0, 1].bar(SAT_ORDER, sat_rate["Attrition_Rate_%"], color=ACCENT)
axes[0, 1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1)
axes[0, 1].set_title("Attrition Rate by Job Satisfaction")
axes[0, 1].set_ylabel("Attrition Rate (%)")
for x, v in enumerate(sat_rate["Attrition_Rate_%"]):
    axes[0, 1].text(x, v + 0.4, f"{v:.1f}%", ha="center", fontsize=9)
axes[0, 1].set_ylim(0, sat_rate["Attrition_Rate_%"].max() * 1.22)

wlb_counts = df.WorkLifeBalance_Label.value_counts().reindex(WLB_ORDER)
axes[1, 0].bar(WLB_ORDER, wlb_counts.values, color=NEUTRAL)
axes[1, 0].set_title("Work-Life Balance Distribution (Bad to Best)")
axes[1, 0].set_ylabel("Number of Employees")
for x, v in enumerate(wlb_counts.values):
    axes[1, 0].text(x, v + 40, f"{v:,}", ha="center", fontsize=9)
axes[1, 0].set_ylim(0, wlb_counts.max() * 1.15)

wlb_rate = attrition_summary(df, "WorkLifeBalance_Label").reindex(WLB_ORDER)
axes[1, 1].bar(WLB_ORDER, wlb_rate["Attrition_Rate_%"], color=ACCENT)
axes[1, 1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1)
axes[1, 1].set_title("Attrition Rate by Work-Life Balance")
axes[1, 1].set_ylabel("Attrition Rate (%)")
for x, v in enumerate(wlb_rate["Attrition_Rate_%"]):
    axes[1, 1].text(x, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
axes[1, 1].set_ylim(0, wlb_rate["Attrition_Rate_%"].max() * 1.22)

plt.tight_layout(); plt.show()

display(sat_rate); display(wlb_rate)

**Interpretation.**

- **Job satisfaction** shows a broadly monotonic association: Low satisfaction employees leave at
  **22.91%** versus 11.40% for Very High — roughly double. The middle two categories sit close
  together (16.5%), so the signal is concentrated at the extremes rather than spread evenly across
  the scale.
- **Work-life balance** shows the sharpest single-category effect: employees rating their balance
  **Bad** leave at **31.22%**, almost double the company average. But the relationship is **not
  monotonic** — the "Best" category (17.96%) has *higher* attrition than "Better" (14.41%). This is a
  genuinely interesting anomaly. One plausible reading is that employees reporting the maximum
  balance include part-time or disengaged staff; another is simple sampling variation in a
  451-employee group. The data cannot distinguish between these, and the notebook does not guess.

## 9.1 Combined Analysis: Job Satisfaction × Work-Life Balance × Attrition

In [ ]:
pivot_rate = (df.pivot_table(index="JobSatisfaction_Label", columns="WorkLifeBalance_Label",
                             values="Attrition_Flag", aggfunc="mean", observed=True) * 100
              ).reindex(index=SAT_ORDER, columns=WLB_ORDER)
pivot_n = (df.pivot_table(index="JobSatisfaction_Label", columns="WorkLifeBalance_Label",
                          values="Attrition_Flag", aggfunc="size", observed=True)
           ).reindex(index=SAT_ORDER, columns=WLB_ORDER)

annotations = pivot_rate.round(1).astype(str) + "%\n(n=" + pivot_n.astype(int).astype(str) + ")"

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_rate, annot=annotations, fmt="", cmap="Reds", ax=ax,
            cbar_kws={"label": "Attrition Rate (%)"}, linewidths=0.6, annot_kws={"size": 9})
ax.set_title("Attrition Rate: Job Satisfaction x Work-Life Balance")
ax.set_xlabel("Work-Life Balance (Bad to Best)"); ax.set_ylabel("Job Satisfaction")
plt.tight_layout(); plt.show()

print("Cells with fewer than 100 employees should be read with caution:")
display(pivot_n)

**Interpretation.** The heatmap shows the two factors compounding. The worst cell — Low
satisfaction combined with Bad work-life balance — carries a dramatically elevated rate, while the
best-performing cells sit well below the 16.2% company average. The employee-count grid is displayed
alongside deliberately: several corner cells contain fewer than 100 employees, and their rates are
correspondingly unstable.

**Association, not causation — restated concretely.** A 40%+ attrition rate in the low-satisfaction /
poor-balance cell does **not** mean that improving those two scores would cut attrition to the
company average. The same employees may be concentrated in early tenure, frequent travel or the HR
function, all of which independently show elevated attrition. Disentangling those overlapping
explanations requires multivariate modelling, which is outside the scope of this descriptive EDA and
is flagged as a recommended next step in Section 25.

---

# 10. Performance Analysis

**Objective:** examine how performance ratings are distributed and whether performance relates to
compensation, satisfaction or attrition.

Note that `PerformanceRating` takes only two values in this dataset (3 = Excellent, 4 = Outstanding).
There are no low performers recorded, which limits what can be concluded: this is a comparison
between *good* and *very good*, not between high and low performers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

perf_counts = df.PerformanceRating_Label.value_counts()
axes[0].bar(perf_counts.index, perf_counts.values, color=NEUTRAL)
axes[0].set_title("Performance Rating Distribution"); axes[0].set_ylabel("Number of Employees")
for x, v in enumerate(perf_counts.values):
    axes[0].text(x, v + 50, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=9)
axes[0].set_ylim(0, perf_counts.max() * 1.2)

perf_rate = attrition_summary(df, "PerformanceRating_Label")
axes[1].bar(perf_rate.index, perf_rate["Attrition_Rate_%"], color=ACCENT)
axes[1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company avg {COMPANY_ATTRITION_RATE:.1f}%")
axes[1].set_title("Attrition Rate by Performance Rating"); axes[1].set_ylabel("Attrition Rate (%)")
axes[1].legend(fontsize=9)
for x, v in enumerate(perf_rate["Attrition_Rate_%"]):
    axes[1].text(x, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)
axes[1].set_ylim(0, perf_rate["Attrition_Rate_%"].max() * 1.25)

sns.boxplot(data=df, x="PerformanceRating_Label", y="PercentSalaryHike",
            color=NEUTRAL, ax=axes[2], fliersize=2)
axes[2].set_title("Salary Hike by Performance Rating")
axes[2].set_xlabel(""); axes[2].set_ylabel("Salary Hike (%)")

plt.tight_layout(); plt.show()

perf_table = (df.groupby("PerformanceRating_Label")
                .agg(Employees=("Attrition_Flag", "size"),
                     Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                     Avg_Salary_Hike=("PercentSalaryHike", "mean"),
                     Median_Income=("MonthlyIncome", "median"),
                     Avg_Job_Satisfaction=("JobSatisfaction", "mean"),
                     Avg_Training=("TrainingTimesLastYear", "mean")).round(2))
display(perf_table)

**Interpretation.** 85% of employees are rated Excellent and 15% Outstanding — no employee is
rated below 3, so the organisation's rating distribution is compressed at the top (a common
real-world pattern, and a known limitation of this variable).

The one strong, coherent relationship is **performance rating → salary hike**: Outstanding performers
average a 21.85% hike versus 14.00% for Excellent (r ≈ 0.77). The pay-for-performance mechanism is
clearly operating and internally consistent.

Attrition is marginally *higher* among Outstanding performers (18.35% vs 15.81%), but the gap is
small and the Outstanding group is far smaller. Median income and average job satisfaction are
near-identical across the two ratings — notably, being rated Outstanding raises the *hike* but not
the *income level*, another reflection of the compensation-structure caveat from Section 3.

## 10.1 Performance vs Career Progression

In [ ]:
perf_career = (df.groupby(["PerformanceRating_Label", "Promotion_Status"], observed=True)
                 .agg(Employees=("Attrition_Flag", "size"),
                      Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                      Avg_Years_Since_Promotion=("YearsSinceLastPromotion", "mean"),
                      Avg_Tenure=("YearsAtCompany", "mean")).round(2))
display(perf_career)

**Interpretation.** The classic "high performers blocked from promotion leave first" pattern
would show Outstanding performers with a long promotion gap leaving most. It does not appear here:
that group sits at 19.60% while Outstanding performers with **no** promotion gap sit at 16.48%.

The wider spread is on the Excellent side — 19.47% with no promotion gap recorded versus 13.43% with
a gap of a year or more. That looks counter-intuitive until the tenure column is read: employees with
no promotion gap recorded average **4.4 years** of tenure against **8.8 years** for those with a gap.
The "no gap" group is simply the newer population, and Section 7.4 already established that early
tenure carries the highest attrition. This is a tenure effect wearing a promotion label, not a
promotion finding.

With only two performance levels available, any performance-related test in this dataset is weak.

---

# 11. Career Growth & Promotion Analysis

**Objective:** examine career-progression variables and their relationship with attrition.

**Variable caveat.** `YearsInCurrentRole` is not present in this dataset (see Section 2 audit).
Additionally, `YearsSinceLastPromotion = 0` is ambiguous: it may mean "promoted within the last year"
*or* "no promotion event ever recorded". Language throughout this section is kept neutral
accordingly — the notebook does not claim these 1,710 employees were "never promoted".

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].hist(df.YearsSinceLastPromotion, bins=16, color=NEUTRAL, edgecolor="white")
axes[0].set_title("Years Since Last Promotion"); axes[0].set_xlabel("Years")
axes[0].set_ylabel("Number of Employees")

axes[1].hist(df.YearsWithCurrManager, bins=18, color=NEUTRAL, edgecolor="white")
axes[1].set_title("Years With Current Manager"); axes[1].set_xlabel("Years")
axes[1].set_ylabel("Number of Employees")

promo_rate = attrition_summary(df, "Promotion_Status")
axes[2].bar(promo_rate.index.astype(str), promo_rate["Attrition_Rate_%"], color=ACCENT)
axes[2].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1)
axes[2].set_title("Attrition Rate by Promotion Status"); axes[2].set_ylabel("Attrition Rate (%)")
axes[2].tick_params(axis="x", labelrotation=12)
for x, (v, n) in enumerate(zip(promo_rate["Attrition_Rate_%"], promo_rate.Employees)):
    axes[2].text(x, v + 0.3, f"{v:.1f}%\n(n={n:,})", ha="center", fontsize=9)
axes[2].set_ylim(0, promo_rate["Attrition_Rate_%"].max() * 1.3)

plt.tight_layout(); plt.show()

print(f"Employees with no promotion gap recorded: {(df.YearsSinceLastPromotion == 0).sum():,} "
      f"({(df.YearsSinceLastPromotion == 0).mean()*100:.1f}%)")
display(promo_rate)

In [ ]:
# Attrition rate across the promotion-gap scale, with small groups consolidated
promo_bins = pd.cut(df.YearsSinceLastPromotion, bins=[-1, 0, 1, 2, 3, 5, 7, np.inf],
                    labels=["0", "1", "2", "3", "4-5", "6-7", "8+"])
promo_scale = attrition_summary(df.assign(Promo_Years=promo_bins), "Promo_Years").sort_index()
display(promo_scale)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(promo_scale.index.astype(str), promo_scale["Attrition_Rate_%"], color=ACCENT)
ax.axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
           label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
ax.set_title("Attrition Rate by Years Since Last Promotion")
ax.set_xlabel("Years Since Last Promotion"); ax.set_ylabel("Attrition Rate (%)")
ax.legend(fontsize=9)
for x, (v, n) in enumerate(zip(promo_scale["Attrition_Rate_%"], promo_scale.Employees)):
    ax.text(x, v + 0.3, f"{v:.1f}%\n(n={n:,})", ha="center", fontsize=8)
ax.set_ylim(0, promo_scale["Attrition_Rate_%"].max() * 1.28)
plt.tight_layout(); plt.show()

In [ ]:
# Manager relationship length vs attrition
mgr_bins = pd.cut(df.YearsWithCurrManager, bins=[-1, 0, 2, 4, 7, np.inf],
                  labels=["0", "1-2", "3-4", "5-7", "8+"])
mgr_summary = attrition_summary(df.assign(Mgr_Years=mgr_bins), "Mgr_Years").sort_index()
display(mgr_summary)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mgr_summary.index.astype(str), mgr_summary["Attrition_Rate_%"], color=ACCENT)
ax.axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
           label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
ax.set_title("Attrition Rate by Years With Current Manager")
ax.set_xlabel("Years With Current Manager"); ax.set_ylabel("Attrition Rate (%)")
ax.legend(fontsize=9)
for x, (v, n) in enumerate(zip(mgr_summary["Attrition_Rate_%"], mgr_summary.Employees)):
    ax.text(x, v + 0.4, f"{v:.1f}%\n(n={n:,})", ha="center", fontsize=8)
ax.set_ylim(0, mgr_summary["Attrition_Rate_%"].max() * 1.25)
plt.tight_layout(); plt.show()

**Interpretation.** Two findings, pulling in different directions:

1. **Promotion gap shows no clear attrition gradient — the pattern is erratic.** Rates by gap
   length run 19.01% (0 years), 13.69% (1), 17.06% (2), 17.42% (3), **6.80%** (4–5), **20.70%** (6–7)
   and 12.30% (8+). Adjacent bands swing by 14 points in both directions, and employees with the
   longest gaps are *not* the highest-risk group. Whatever is driving exits here, a long wait for
   promotion is not visibly it.

2. **Time with current manager shows a much cleaner signal.** Employees in their first year under a
   manager leave at **32.86%**, against 14.47% at 1–2 years and **8.16%** at 8+ years. The decline is
   not perfectly monotonic — the 5–7 year band (14.25%) sits slightly above 3–4 years (12.50%) — but
   the first-year figure stands out sharply from everything after it.

Finding 2 must be read carefully, because it is substantially **confounded with tenure**:
`YearsWithCurrManager` correlates 0.77 with `YearsAtCompany`. New employees necessarily have new
managers. The manager-relationship effect and the early-tenure effect cannot be separated with this
data — they may be the same phenomenon viewed twice. Section 17 examines the interaction directly.

---

# 12. Experience & Tenure Analysis

**Objective:** examine career experience and company tenure — the variables that Section 7 identified
as producing the widest attrition spreads — and their relationships with income, satisfaction and
job level.

In [ ]:
tenure_vars = ["TotalWorkingYears", "YearsAtCompany", "YearsSinceLastPromotion",
               "YearsWithCurrManager", "NumCompaniesWorked"]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.8))
for ax, var in zip(axes, tenure_vars):
    sns.boxplot(data=df, x="Attrition", y=var, hue="Attrition", palette=ATTRITION_PALETTE,
                legend=False, ax=ax, fliersize=2)
    ax.set_title(var, fontsize=10.5); ax.set_xlabel(""); ax.set_ylabel("Years" if var != "NumCompaniesWorked" else "Count")
plt.suptitle("Experience & Tenure Variables by Attrition Status", y=1.03, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

comparison = (df.groupby("Attrition")[tenure_vars].median()
                .T.rename(columns={"No": "Stayed_Median", "Yes": "Left_Median"}))
comparison["Difference"] = comparison.Left_Median - comparison.Stayed_Median
display(comparison)

**Interpretation.** Leavers are consistently *earlier* in every tenure measure. Median total
working years is 7 for leavers against 10 for stayers; median company tenure is 3 against 6; median
years with current manager is 2 against 3. `NumCompaniesWorked` shows almost no difference, so there
is no evidence here of a "job-hopper" profile driving exits.

The picture is coherent: **this organisation's attrition is concentrated among employees who are
early in their careers and early in their tenure**, not among long-serving staff.

In [ ]:
# How tenure relates to income, satisfaction and job level
tenure_profile = (df.groupby("Tenure_Group", observed=True)
                    .agg(Employees=("Attrition_Flag", "size"),
                         Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                         Median_Income=("MonthlyIncome", "median"),
                         Avg_Job_Satisfaction=("JobSatisfaction", "mean"),
                         Avg_Job_Level=("JobLevel", "mean"),
                         Avg_Total_Experience=("TotalWorkingYears", "mean")).round(2))
display(tenure_profile)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].plot(tenure_profile.index.astype(str), tenure_profile.Attrition_Rate,
             marker="o", color=ACCENT, lw=2)
axes[0].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
axes[0].set_title("Attrition Rate Across Tenure Bands")
axes[0].set_xlabel("Years at Company"); axes[0].set_ylabel("Attrition Rate (%)")
axes[0].legend(fontsize=9)
for x, v in enumerate(tenure_profile.Attrition_Rate):
    axes[0].text(x, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

exp_profile = (df.groupby("Experience_Group", observed=True)
                 .Attrition_Flag.mean().mul(100).round(2))
axes[1].plot(exp_profile.index.astype(str), exp_profile.values, marker="o", color=NEUTRAL, lw=2)
axes[1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
axes[1].set_title("Attrition Rate Across Career-Experience Bands")
axes[1].set_xlabel("Total Working Years"); axes[1].set_ylabel("Attrition Rate (%)")
axes[1].legend(fontsize=9)
for x, v in enumerate(exp_profile.values):
    axes[1].text(x, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout(); plt.show()

**Interpretation.** Both curves fall steeply then flatten — attrition risk is concentrated in
the first two to five years and then stabilises at a low level. The 11–20 year band is the most
stable segment in the organisation at 6.79%, with a slight uptick in the 20+ band (12.24%) that is
plausibly retirement-related rather than resignation-related, though the data cannot distinguish
between exit types.

Two further columns deserve attention, and they point the same way. Median income barely moves
across tenure bands (47.7K → 53.4K → 43.9K, and not in order). **Average job level does not rise with
tenure either** — it runs 2.07, 2.16, 2.07, 1.95, 1.78, drifting slightly *downward* in the
longest-serving bands. In a real organisation, both would climb with tenure.

This is further evidence for the structural caveat established in Section 3: seniority, pay and
tenure are not linked in this dataset the way they are in a real HRIS. Career-progression conclusions
are therefore avoided, while the attrition-by-tenure finding — which depends on none of these
fields — stands on its own.

---

# 13. Training & Development

**Objective:** examine training frequency and its relationship with performance, attrition and
career progression.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

train_counts = df.TrainingTimesLastYear.value_counts().sort_index()
axes[0].bar(train_counts.index.astype(str), train_counts.values, color=NEUTRAL)
axes[0].set_title("Training Sessions Last Year"); axes[0].set_xlabel("Number of Sessions")
axes[0].set_ylabel("Number of Employees")
for x, v in enumerate(train_counts.values):
    axes[0].text(x, v + 25, f"{v:,}", ha="center", fontsize=8)
axes[0].set_ylim(0, train_counts.max() * 1.15)

train_dept = df.groupby("Department").TrainingTimesLastYear.mean().sort_values()
axes[1].barh(train_dept.index, train_dept.values, color=NEUTRAL)
axes[1].set_title("Average Training Sessions by Department")
axes[1].set_xlabel("Average Sessions per Employee")
for y, v in enumerate(train_dept.values):
    axes[1].text(v + 0.03, y, f"{v:.2f}", va="center", fontsize=9)
axes[1].set_xlim(0, train_dept.max() * 1.18)

train_rate = attrition_summary(df, "TrainingTimesLastYear").sort_index()
axes[2].bar(train_rate.index.astype(str), train_rate["Attrition_Rate_%"], color=ACCENT)
axes[2].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company avg {COMPANY_ATTRITION_RATE:.1f}%")
axes[2].set_title("Attrition Rate by Training Frequency")
axes[2].set_xlabel("Training Sessions Last Year"); axes[2].set_ylabel("Attrition Rate (%)")
axes[2].legend(fontsize=9)
for x, (v, n) in enumerate(zip(train_rate["Attrition_Rate_%"], train_rate.Employees)):
    axes[2].text(x, v + 0.4, f"{v:.1f}%", ha="center", fontsize=8)
axes[2].set_ylim(0, train_rate["Attrition_Rate_%"].max() * 1.25)

plt.tight_layout(); plt.show()
display(train_rate)

In [ ]:
training_profile = (df.groupby("TrainingTimesLastYear")
                      .agg(Employees=("Attrition_Flag", "size"),
                           Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                           Pct_Outstanding=("PerformanceRating", lambda s: round((s == 4).mean() * 100, 2)),
                           Avg_Job_Satisfaction=("JobSatisfaction", "mean"),
                           Avg_Tenure=("YearsAtCompany", "mean"),
                           Avg_Salary_Hike=("PercentSalaryHike", "mean")).round(2))
display(training_profile)

**Interpretation.** Training is concentrated at 2–3 sessions per year (71% of employees), and
average training volume is nearly identical across departments (2.62–2.81 sessions) — a uniform
policy rather than a differentiated investment.

The attrition pattern is **not monotonic**, which is worth stating plainly: employees with zero
training show the highest rate (18.63%), and those with 6 sessions the lowest (6.32%) — but the
groups in between do not line up in order, and the 6-session group contains only 190 employees.

The tempting business story — "training retains people" — is **not supported** by this evidence.
An equally consistent explanation runs the other way: employees already disengaged or on their way
out attend less training. With a single snapshot of cross-sectional data, the direction of this
relationship cannot be determined.

The percentage of Outstanding performers shows no meaningful gradient across training frequency
either, so no training-to-performance link is evidenced here.

---

# 14. Business Travel Analysis

**Objective:** examine business travel frequency against attrition and across departments.

### Overtime Analysis — Unavailable

The project plan called for an overtime analysis (overtime distribution, overtime vs attrition,
overtime vs job satisfaction, overtime by role). **The `OverTime` column does not exist in this
dataset.** No substitute variable measures hours worked, so this analysis is reported as unavailable
rather than approximated with an unrelated proxy. It is listed in Section 25 as a data-collection
recommendation.

In [ ]:
travel_summary = attrition_summary(df, "BusinessTravel")
display(travel_summary)
plot_attrition_rate(travel_summary, "Attrition Rate by Business Travel Frequency",
                    benchmark=COMPANY_ATTRITION_RATE, figsize=(10, 3.4))

In [ ]:
TRAVEL_ORDER = ["Non-Travel", "Travel_Rarely", "Travel_Frequently"]

travel_dept = (df.pivot_table(index="Department", columns="BusinessTravel",
                             values="Attrition_Flag", aggfunc="mean", observed=True) * 100
               ).reindex(columns=TRAVEL_ORDER).round(1)
travel_dept_n = (df.pivot_table(index="Department", columns="BusinessTravel",
                               values="Attrition_Flag", aggfunc="size", observed=True)
                 ).reindex(columns=TRAVEL_ORDER)

annot = travel_dept.astype(str) + "%\n(n=" + travel_dept_n.astype(int).astype(str) + ")"

fig, ax = plt.subplots(figsize=(9.5, 3.6))
sns.heatmap(travel_dept, annot=annot, fmt="", cmap="Reds", ax=ax, linewidths=0.6,
            cbar_kws={"label": "Attrition Rate (%)"}, annot_kws={"size": 9})
ax.set_title("Attrition Rate: Department x Business Travel")
ax.set_xlabel("Business Travel Frequency"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
travel_profile = (df.groupby("BusinessTravel")
                    .agg(Employees=("Attrition_Flag", "size"),
                         Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                         Avg_Job_Satisfaction=("JobSatisfaction", "mean"),
                         Avg_Work_Life_Balance=("WorkLifeBalance", "mean"),
                         Median_Income=("MonthlyIncome", "median"),
                         Avg_Tenure=("YearsAtCompany", "mean")).round(2)
                    .reindex(TRAVEL_ORDER))
display(travel_profile)

**Interpretation.** Business travel produces one of the cleanest monotonic patterns in the
entire analysis: 8.14% → 15.08% → 24.79% as travel frequency rises. Frequent travellers leave at
roughly three times the rate of non-travellers, and the pattern **holds in all three departments**,
which is important — it means the effect is not an artefact of one department that happens to travel
more.

The profile table adds a useful nuance: average job satisfaction (2.70–2.79) and work-life
balance (2.76–2.78) scores are nearly **identical** across the three travel groups. Frequent travellers are not reporting worse
satisfaction or worse balance, yet they leave far more often. Whatever is happening, it is not
being captured by the existing satisfaction survey instrument — itself a finding worth passing to HR.

Consistency across departments and a clean dose-response shape make this the most robust
association in the dataset. It still is not causation: travel-heavy roles may differ systematically
in ways not measured here (client-facing pressure, external recruiter exposure, relocation).

---

# 15. Correlation Analysis

**Objective:** measure linear relationships among the numerical and ordinal variables to identify
redundancy, confirm expected structure, and flag which variables carry independent information.

**Two methodological cautions applied here:**

1. Ordinal 1–4 rating scales are included because they are meaningfully ordered, but Pearson
   correlation on a 4-point scale is a weak instrument and its magnitudes are not comparable to
   those between continuous variables.
2. **Correlation is not causation, and a correlation near zero is not "no relationship"** — it means
   no *linear* relationship. Non-linear patterns (like the U-shaped work-life balance effect seen in
   Section 9) are invisible to this matrix.

In [ ]:
CORRELATION_VARS = [
    "Age", "MonthlyIncome", "TotalWorkingYears", "YearsAtCompany",
    "YearsSinceLastPromotion", "YearsWithCurrManager", "NumCompaniesWorked",
    "DistanceFromHome", "JobLevel", "JobSatisfaction", "EnvironmentSatisfaction",
    "WorkLifeBalance", "JobInvolvement", "PerformanceRating", "PercentSalaryHike",
    "TrainingTimesLastYear", "StockOptionLevel", "Attrition_Flag",
]

corr = df[CORRELATION_VARS].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.4, ax=ax, annot_kws={"size": 7.5},
            cbar_kws={"label": "Pearson correlation"})
ax.set_title("Correlation Matrix: HR Numerical & Ordinal Variables")
plt.tight_layout(); plt.show()

In [ ]:
# Extract the strongest pairwise relationships (excluding self-correlation)
pairs = (corr.where(~np.eye(len(corr), dtype=bool))
             .stack().drop_duplicates()
             .sort_values(key=abs, ascending=False))
strongest = pairs.head(10).to_frame("Correlation").round(3)
strongest.index = [f"{a}  <->  {b}" for a, b in strongest.index]
print("Ten strongest relationships in the dataset:")
display(strongest)

print("\nCorrelation of each variable with Attrition_Flag:")
attrition_corr = (corr["Attrition_Flag"].drop("Attrition_Flag")
                    .sort_values().to_frame("Correlation_with_Attrition").round(3))
display(attrition_corr)

**Interpretation.**

**Structurally expected and confirmed:**
- `PercentSalaryHike` ↔ `PerformanceRating` (r = 0.77) — the pay-for-performance mechanism.
- `YearsAtCompany` ↔ `YearsWithCurrManager` (r = 0.77) — you cannot have known your manager longer
  than you have worked here.
- `Age` ↔ `TotalWorkingYears` (r = 0.68) — career length tracks age.
- `TotalWorkingYears` ↔ `YearsAtCompany` (r = 0.63).

These high correlations are a **redundancy warning**: the tenure variables largely measure the same
underlying construct. Treating them as four independent risk factors would multiply-count a single
phenomenon.

**Structurally unexpected:** `MonthlyIncome` correlates with essentially nothing — 0.05 with
`JobLevel`, −0.03 with `TotalWorkingYears`. This is the quantitative form of the caveat raised in
Section 3, and it is the reason this notebook draws no compensation conclusions.

**Correlations with attrition are all weak in magnitude** — the strongest is `TotalWorkingYears` at
−0.17. This is expected and **not** a sign the analysis has failed: attrition is a binary outcome,
and a point-biserial correlation of −0.17 on a binary target is a real, usable signal. It is also a
reminder that **no single variable explains attrition in this dataset** — the segment-level rate
differences seen in Section 7 (a 4.5× spread across tenure bands) tell the story far better than any
correlation coefficient.

The ordering is nonetheless informative: experience, manager tenure, age and company tenure sit at
the top; `DistanceFromHome`, `StockOptionLevel` and `JobLevel` are effectively flat.

---

# 16. Multivariate Analysis

**Objective:** examine how factors combine. Single-variable attrition rates can mislead when two
variables overlap — for instance, if frequent travellers are disproportionately early-tenure
employees, some of the "travel effect" is really the tenure effect. Cross-tabulation is the simplest
honest way to check whether an effect survives when another variable is held roughly constant.

Only combinations that revealed a usable pattern are retained below.

## 16.1 Age × Income × Attrition

In [ ]:
age_income = (df.pivot_table(index="Age_Group", columns="Income_Band",
                             values="Attrition_Flag", aggfunc="mean", observed=True) * 100).round(1)
age_income_n = df.pivot_table(index="Age_Group", columns="Income_Band",
                              values="Attrition_Flag", aggfunc="size", observed=True)

fig, ax = plt.subplots(figsize=(10, 4.4))
sns.heatmap(age_income, annot=age_income.astype(str) + "%\n(n=" + age_income_n.astype(int).astype(str) + ")",
            fmt="", cmap="Reds", linewidths=0.6, ax=ax,
            cbar_kws={"label": "Attrition Rate (%)"}, annot_kws={"size": 8.5})
ax.set_title("Attrition Rate: Age Group x Income Band")
ax.set_xlabel("Income Band"); ax.set_ylabel("Age Group")
plt.tight_layout(); plt.show()

**Interpretation.** Reading **down each column** (age varying, income held constant), rates fall
sharply and consistently — every income quartile shows roughly 30–39% attrition at ages 18–24 against
7–13% at ages 35–44. Reading **across each row** (income varying, age held constant), no consistent
direction emerges.

The age effect therefore survives controlling for income; the income effect largely does not.
**Age is doing the work here, not pay.**

Two cells warrant caution: the 18–24 / Q3 cell (63.2%) and the 55+ / Q2 cell (33.3%) both rest on
very small counts, visible in the annotations. They are noise, not a sub-pattern.

## 16.2 Tenure × Job Satisfaction × Attrition

In [ ]:
tenure_sat = (df.pivot_table(index="Tenure_Group", columns="JobSatisfaction_Label",
                             values="Attrition_Flag", aggfunc="mean", observed=True) * 100
              ).reindex(columns=SAT_ORDER).round(1)
tenure_sat_n = (df.pivot_table(index="Tenure_Group", columns="JobSatisfaction_Label",
                               values="Attrition_Flag", aggfunc="size", observed=True)
                ).reindex(columns=SAT_ORDER)

fig, ax = plt.subplots(figsize=(10, 4.4))
sns.heatmap(tenure_sat, annot=tenure_sat.astype(str) + "%\n(n=" + tenure_sat_n.astype(int).astype(str) + ")",
            fmt="", cmap="Reds", linewidths=0.6, ax=ax,
            cbar_kws={"label": "Attrition Rate (%)"}, annot_kws={"size": 8.5})
ax.set_title("Attrition Rate: Company Tenure x Job Satisfaction")
ax.set_xlabel("Job Satisfaction"); ax.set_ylabel("Years at Company")
plt.tight_layout(); plt.show()

**Interpretation.** Both effects persist when the other is held constant, and they compound. The
worst cell — 0–2 years tenure combined with Low satisfaction — reaches **43.1%**, while long-tenure
employees with mid-to-high satisfaction sit in the 3–7% range.

The most important cell is easy to miss: **employees with Very High satisfaction in their first two
years still leave at 19.5%** — above the 16.2% company average. Satisfaction does not neutralise the
early-tenure effect. Whatever is driving first-two-year exits is not fully captured by how satisfied
people say they are.

The 20+ year row reverses direction (7.1% at Low satisfaction, 16.1% at Very High) on small counts.
That is almost certainly noise, and possibly retirement-related exits, which the `Attrition` flag
cannot distinguish.

## 16.3 Business Travel × Tenure × Attrition — Does the Travel Effect Survive?

In [ ]:
travel_tenure = (df.pivot_table(index="BusinessTravel", columns="Tenure_Group",
                                values="Attrition_Flag", aggfunc="mean", observed=True) * 100
                 ).reindex(index=TRAVEL_ORDER).round(1)
travel_tenure_n = (df.pivot_table(index="BusinessTravel", columns="Tenure_Group",
                                  values="Attrition_Flag", aggfunc="size", observed=True)
                   ).reindex(index=TRAVEL_ORDER)

fig, ax = plt.subplots(figsize=(10, 3.8))
sns.heatmap(travel_tenure,
            annot=travel_tenure.astype(str) + "%\n(n=" + travel_tenure_n.astype(int).astype(str) + ")",
            fmt="", cmap="Reds", linewidths=0.6, ax=ax,
            cbar_kws={"label": "Attrition Rate (%)"}, annot_kws={"size": 8.5})
ax.set_title("Attrition Rate: Business Travel x Company Tenure")
ax.set_xlabel("Years at Company"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

print("Tenure composition within each travel group (%) - checking for confounding:")
display((pd.crosstab(df.BusinessTravel, df.Tenure_Group, normalize="index") * 100)
        .reindex(TRAVEL_ORDER).round(1))

**Interpretation — this is the key confounding check.** The composition table shows the three
travel groups have **broadly similar tenure profiles**: 20.6% / 23.4% / 23.4% of each group sits in
the 0–2 year band. Frequent travellers are not disproportionately new employees, so the travel effect
cannot be dismissed as a tenure effect in disguise.

And the gradient appears **inside every single tenure band**:

| Tenure | Non-Travel | Travel_Rarely | Travel_Frequently |
|---|---|---|---|
| 0–2 years | 19.8% | 26.6% | **48.7%** |
| 3–5 years | 9.0% | 12.6% | 23.3% |
| 6–10 years | 4.6% | 11.9% | 16.4% |
| 11–20 years | 0.0% | 7.4% | 9.7% |
| 20+ years | 0.0% | 13.2% | 13.3% |

The direction is identical in all five bands. The combination of early tenure *and* frequent travel
produces the highest rate found anywhere in this analysis — **48.7%**, three times the company
average.

The travel–attrition association therefore **survives controlling for tenure**. That is a
meaningfully stronger result than the raw single-variable rate, and it is why business travel is
treated as the most robust finding in this analysis.

## 16.4 Job Role × Business Travel × Attrition

In [ ]:
role_travel = (df.pivot_table(index="JobRole", columns="BusinessTravel",
                              values="Attrition_Flag", aggfunc="mean", observed=True) * 100
               ).reindex(columns=TRAVEL_ORDER).round(1)
role_travel_n = (df.pivot_table(index="JobRole", columns="BusinessTravel",
                                values="Attrition_Flag", aggfunc="size", observed=True)
                 ).reindex(columns=TRAVEL_ORDER)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(role_travel,
            annot=role_travel.astype(str) + "%\n(n=" + role_travel_n.astype(int).astype(str) + ")",
            fmt="", cmap="Reds", linewidths=0.6, ax=ax,
            cbar_kws={"label": "Attrition Rate (%)"}, annot_kws={"size": 8})
ax.set_title("Attrition Rate: Job Role x Business Travel")
ax.set_xlabel("Business Travel Frequency"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

**Interpretation.** The travel gradient reappears within most job roles, reinforcing 16.3.
Some cells contain fewer than 40 employees (visible in the annotations) and should not be read as
precise role-level estimates — the value of this view is the consistency of the direction, not the
individual numbers.

## 16.5 Combined Risk Factor Count

A simple additive check: how does attrition change as an employee accumulates the risk markers
identified in earlier sections?

In [ ]:
risk_factors = pd.DataFrame({
    "Early_Tenure": (df.YearsAtCompany < 3).astype(int),
    "Frequent_Travel": (df.BusinessTravel == "Travel_Frequently").astype(int),
    "Low_Satisfaction": (df.JobSatisfaction <= 2).astype(int),
    "Poor_Work_Life_Balance": (df.WorkLifeBalance <= 2).astype(int),
    "Young_Age": (df.Age < 30).astype(int),
})
df["Risk_Factor_Count"] = risk_factors.sum(axis=1)

factor_summary = attrition_summary(df, "Risk_Factor_Count").sort_index()
display(factor_summary)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(factor_summary.index.astype(str), factor_summary["Attrition_Rate_%"], color=ACCENT)
ax.axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
           label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
ax.set_title("Attrition Rate by Number of Risk Markers Present")
ax.set_xlabel("Number of Risk Markers (of 5)"); ax.set_ylabel("Attrition Rate (%)")
ax.legend(fontsize=9)
for x, (v, n) in enumerate(zip(factor_summary["Attrition_Rate_%"], factor_summary.Employees)):
    ax.text(x, v + 0.8, f"{v:.1f}%\n(n={n:,})", ha="center", fontsize=9)
ax.set_ylim(0, factor_summary["Attrition_Rate_%"].max() * 1.25)
plt.tight_layout(); plt.show()

**Interpretation.** Attrition climbs **strictly monotonically** with the number of markers
present: 6.16% (0 markers, n=926) → 12.39% (1) → 18.60% (2) → 36.93% (3, n=501) → 55.56% (4, n=54) →
100% (5, n=9).

That the relationship is perfectly ordered is reassuring — the markers identified independently
across Sections 7–15 point in a consistent direction rather than contradicting each other. The three
lowest counts cover 3,763 employees, so the useful part of this curve is well-populated; the 4- and
5-marker groups are tiny (63 employees between them) and their rates should be read as illustrative
rather than precise. The 100% figure rests on nine employees.

**This is a descriptive summary, not a model.** The markers are not weighted, not tested for
independence, and the highest-count groups contain few employees. It shows that risk *accumulates*;
it does not quantify how much each marker contributes.

---

# 17. Outlier Analysis

**Objective:** identify extreme values in key numerical variables and assess whether each represents
a data error or a legitimate employee record.

**Policy: no outlier is removed.** In HR data, extreme values are usually *real people* — the
40-year veteran, the executive at the top of the pay scale. Removing them would delete exactly the
employees an HR analysis should be most interested in. They are quantified, inspected and retained.

In [ ]:
def iqr_outlier_report(data, column):
    # Flag values beyond 1.5 x IQR from the quartiles and return a summary row.
    q1, q3 = data[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return {
        "Variable": column,
        "Q1": round(q1, 1), "Q3": round(q3, 1), "IQR": round(iqr, 1),
        "Lower_Fence": round(lower, 1), "Upper_Fence": round(upper, 1),
        "Outliers": int(mask.sum()),
        "Pct_of_Rows": round(mask.mean() * 100, 2),
        "Min_Observed": data[column].min(), "Max_Observed": data[column].max(),
    }

OUTLIER_VARS = ["MonthlyIncome", "TotalWorkingYears", "YearsAtCompany",
                "NumCompaniesWorked", "PercentSalaryHike", "Age",
                "YearsSinceLastPromotion", "YearsWithCurrManager"]

outlier_report = pd.DataFrame([iqr_outlier_report(df, v) for v in OUTLIER_VARS])
display(outlier_report)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, var in zip(axes.flat, OUTLIER_VARS):
    sns.boxplot(y=df[var], color=NEUTRAL, ax=ax, fliersize=2.5)
    ax.set_title(var, fontsize=10.5); ax.set_ylabel("")
    if var == "MonthlyIncome":
        thousands_formatter(ax, "y")
plt.suptitle("Outlier Detection: Boxplots for Key Numerical Variables",
             y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Inspect the most extreme records rather than judging them from summary statistics alone
print("Five longest-serving employees:")
display(df.nlargest(5, "YearsAtCompany")[
    ["EmployeeID", "Age", "Department", "JobRole", "JobLevel", "YearsAtCompany",
     "TotalWorkingYears", "MonthlyIncome", "Attrition"]])

print("\nFive highest-paid employees:")
display(df.nlargest(5, "MonthlyIncome")[
    ["EmployeeID", "Age", "Department", "JobRole", "JobLevel", "TotalWorkingYears",
     "MonthlyIncome", "Attrition"]])

### Outlier Assessment & Retention Decisions

| Variable | IQR outliers | Assessment | Decision |
|---|---|---|---|
| `MonthlyIncome` | 332 (7.67%) above 165,623 | The distribution is heavily right-skewed, so the IQR fence flags the entire upper tail. Top earners are plausible senior staff, and the maximum (199,990) is only 1.2× the fence | **Retain — skew artefact, not error** |
| `YearsAtCompany` | 308 (7.12%) above 18 years | Long-service employees, up to 40 years. Verified against `Age` and `TotalWorkingYears` in Section 3 | **Retain — valid extreme values** |
| `YearsSinceLastPromotion` | 317 (7.33%) above 7.5 years | Long promotion gaps are real in flat structures. Worth an HR question, not a deletion | **Retain — potential business exception** |
| `TotalWorkingYears` | 186 (4.30%) above 28.5 years | Consistent with employees aged 50–60. Age cross-check passes for all | **Retain — valid extreme values** |
| `NumCompaniesWorked` | 154 (3.56%) at 9 employers | High but entirely plausible over a full career | **Retain** |
| `YearsWithCurrManager` | 40 (0.92%) above 14.5 years | All verified as ≤ `YearsAtCompany` in Section 3 | **Retain** |
| `PercentSalaryHike` | **0** (range 11–25%) | Tightly bounded; no value reaches the fence. Upper tail explained by performance rating (r = 0.77) | **Retain — business-explained** |
| `Age` | **0** (range 18–60) | Normal working-age population | **Retain** |

Note the pattern: the variables with the most "outliers" are precisely the right-skewed tenure and
income measures, where the IQR rule flags a long legitimate tail rather than genuine anomalies.

**Overall:** every flagged value cross-checks correctly against related variables. The only genuinely
impossible record in the dataset was the single tenure inconsistency identified in Section 3, which
is flagged rather than removed. Removing statistical outliers here would delete the most senior and
longest-serving employees from a retention analysis — the opposite of useful.

---

# 18. HR Risk Segmentation

**Objective:** apply the composite risk rule already defined in the project's SQL layer (Query 30),
so that Python, SQL and the Power BI dashboard segment employees identically.

### Business rule

| Level | Condition |
|---|---|
| **High Risk** | `JobSatisfaction <= 2` **AND** `YearsAtCompany < 3` **AND** `WorkLifeBalance <= 2` |
| **Medium Risk** | `JobSatisfaction <= 2` **OR** `WorkLifeBalance <= 2` |
| **Low Risk** | All other employees |

### What this is — and is not

This is a **transparent, deterministic business rule**: three thresholds combined with AND/OR logic.
Every classification can be traced by hand to the three fields that produced it.

It is **not** a machine-learning model, **not** a prediction, and **not** a statement about any
individual. Employees meeting the predefined analytical criteria for "High Risk" have **not** been
predicted to leave; they simply satisfy three conditions that, at the group level, are associated
with higher observed attrition. The
validation below measures exactly that — whether the rule separates groups by *historical* attrition
rate — and nothing more.

In [ ]:
def assign_hr_risk_level(row):
    # Mirror of SQL Query 30 risk logic, applied row-wise for readability.
    if (row.JobSatisfaction <= 2) and (row.YearsAtCompany < 3) and (row.WorkLifeBalance <= 2):
        return "High Risk"
    if (row.JobSatisfaction <= 2) or (row.WorkLifeBalance <= 2):
        return "Medium Risk"
    return "Low Risk"


df["HR_Risk_Level"] = df.apply(assign_hr_risk_level, axis=1)
RISK_ORDER = ["Low Risk", "Medium Risk", "High Risk"]
RISK_COLORS = {"Low Risk": POSITIVE, "Medium Risk": "#E67E22", "High Risk": ACCENT}

risk_distribution = (df.HR_Risk_Level.value_counts().reindex(RISK_ORDER).to_frame("Employees")
                       .assign(Pct_of_Workforce=lambda d: (d.Employees / len(df) * 100).round(2)))
display(risk_distribution)

In [ ]:
risk_validation = (df.groupby("HR_Risk_Level")
                     .agg(Employees=("Attrition_Flag", "size"),
                          Attrition_Count=("Attrition_Flag", "sum"),
                          Attrition_Rate=("Attrition_Flag", lambda s: round(s.mean() * 100, 2)),
                          Median_Income=("MonthlyIncome", "median"),
                          Median_Tenure=("YearsAtCompany", "median"),
                          Avg_Job_Satisfaction=("JobSatisfaction", "mean"),
                          Avg_Work_Life_Balance=("WorkLifeBalance", "mean"))
                     .reindex(RISK_ORDER).round(2))
display(risk_validation)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].bar(risk_distribution.index, risk_distribution.Employees,
            color=[RISK_COLORS[r] for r in risk_distribution.index])
axes[0].set_title("Workforce Distribution by HR Risk Level")
axes[0].set_ylabel("Number of Employees")
for x, (v, p) in enumerate(zip(risk_distribution.Employees, risk_distribution.Pct_of_Workforce)):
    axes[0].text(x, v + 35, f"{v:,}\n({p}%)", ha="center", fontsize=9)
axes[0].set_ylim(0, risk_distribution.Employees.max() * 1.2)

axes[1].bar(risk_validation.index, risk_validation.Attrition_Rate,
            color=[RISK_COLORS[r] for r in risk_validation.index])
axes[1].axhline(COMPANY_ATTRITION_RATE, color="black", ls="--", lw=1,
                label=f"Company average {COMPANY_ATTRITION_RATE:.1f}%")
axes[1].set_title("Observed Attrition Rate by HR Risk Level")
axes[1].set_ylabel("Attrition Rate (%)"); axes[1].legend(fontsize=9)
for x, v in enumerate(risk_validation.Attrition_Rate):
    axes[1].text(x, v + 0.8, f"{v:.1f}%", ha="center", fontsize=9)
axes[1].set_ylim(0, risk_validation.Attrition_Rate.max() * 1.2)

plt.tight_layout(); plt.show()

**Interpretation — does the rule separate the workforce usefully?**

Yes, directionally. Observed attrition rises cleanly across the three tiers: **12.06% (Low) → 18.51%
(Medium) → 41.38% (High)**. High Risk employees left at roughly **3.4× the rate** of Low Risk
employees, so the rule is picking up a real signal.

Two honest limitations sit alongside that:

1. **The High Risk tier is tiny** — 87 employees, 2.0% of the workforce. The triple-AND condition is
   very restrictive. It identifies only 36 of the 701 actual leavers, meaning **95% of exits came
   from the Medium and Low tiers**. As a targeting tool it is precise but has very low coverage.
2. **The Medium Risk tier captures more than half the workforce** (2,383 employees) at a rate only
   2.3 points above average. A segment that large is not actionable as a priority list.

For operational use, HR would get more value from the multi-marker view in Section 16.5 — which
includes tenure, travel and age — than from this three-field rule alone. The rule's real value in
this project is **consistency**: SQL, Python and Power BI all segment employees identically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

risk_dept = (pd.crosstab(df.Department, df.HR_Risk_Level, normalize="index") * 100)[RISK_ORDER]
risk_dept.plot(kind="barh", stacked=True, ax=axes[0],
               color=[RISK_COLORS[r] for r in RISK_ORDER], width=0.7)
axes[0].set_title("HR Risk Composition by Department"); axes[0].set_xlabel("Share of Department (%)")
axes[0].set_ylabel(""); axes[0].legend(title="", fontsize=8, loc="lower right")

risk_role = (pd.crosstab(df.JobRole, df.HR_Risk_Level, normalize="index") * 100)[RISK_ORDER]
risk_role = risk_role.sort_values("High Risk")
risk_role.plot(kind="barh", stacked=True, ax=axes[1],
               color=[RISK_COLORS[r] for r in RISK_ORDER], width=0.7)
axes[1].set_title("HR Risk Composition by Job Role"); axes[1].set_xlabel("Share of Role (%)")
axes[1].set_ylabel(""); axes[1].legend(title="", fontsize=8, loc="lower right")

plt.tight_layout(); plt.show()

print("High Risk employees by department (count and share of department):")
display(pd.crosstab(df.Department, df.HR_Risk_Level)[RISK_ORDER])

**Interpretation.** Risk composition is remarkably uniform across departments — each carries
roughly 2–3% High Risk and 54–57% Medium Risk. Human Resources has the largest High Risk share
(3.2%), consistent with its elevated attrition rate, but the difference across departments is small.

This uniformity is itself informative: because the rule uses only satisfaction and tenure fields,
and those fields are distributed similarly across departments, the rule cannot explain the
department-level attrition differences found in Section 7. The variables that *do* differentiate
departments — travel frequency, workforce age profile — are absent from the rule.

---

# 19. Statistical Analysis

**Objective:** test whether the strongest patterns observed in the EDA are statistically
distinguishable from random variation, rather than relying on visual inspection alone.

**Significance level: α = 0.05**, stated explicitly and applied consistently.

**Two principles governing this section:**

1. **Statistical significance ≠ business significance.** With 4,327 records, small and practically
   irrelevant differences can reach significance. Every test below therefore reports an **effect
   size** (Cramér's V for categorical tests, rank-biserial correlation for numerical ones) alongside
   the p-value.
2. **Tests are selected to match the data, not applied reflexively.** Skewed variables are tested
   with Mann-Whitney U rather than a t-test, and both are reported where they disagree — because the
   disagreement is itself informative.

## 19.1 Chi-Square Tests of Independence — Categorical Variables vs Attrition

In [ ]:
ALPHA = 0.05

def chi_square_test(data, column):
    # Chi-square test of independence with Cramer's V effect size.
    table = pd.crosstab(data[column], data.Attrition)
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    effect = ("Negligible" if cramers_v < 0.10 else
              "Small" if cramers_v < 0.20 else
              "Moderate" if cramers_v < 0.30 else "Large")
    return {
        "Variable": column, "Chi2": round(chi2, 2), "df": dof,
        "p_value": p_value,
        "Significant_at_0.05": "Yes" if p_value < ALPHA else "No",
        "Cramers_V": round(cramers_v, 3), "Effect_Size": effect,
        "Min_Expected_Count": round(expected.min(), 1),
    }

CHI_SQUARE_VARS = ["BusinessTravel", "MaritalStatus", "JobRole", "Department",
                   "JobSatisfaction_Label", "WorkLifeBalance_Label",
                   "Gender", "Education_Label", "Age_Group", "Tenure_Group"]

chi_results = pd.DataFrame([chi_square_test(df, v) for v in CHI_SQUARE_VARS])
chi_results["p_value"] = chi_results.p_value.apply(lambda p: f"{p:.2e}")
display(chi_results.sort_values("Cramers_V", ascending=False).reset_index(drop=True))

print(f"All expected cell counts exceed 5, so the chi-square approximation is valid "
      f"(minimum expected count across all tests: {chi_results.Min_Expected_Count.min()}).")

### Chi-Square Test — Full Write-Up

**Research question.** Is attrition independent of an employee's categorical attributes?

**Hypotheses** (applied to each variable separately):
- **H₀:** Attrition status is independent of the variable — the observed differences are due to
  sampling variation.
- **H₁:** Attrition status is associated with the variable.

**Test selected.** Pearson's chi-square test of independence. It is appropriate here because both
variables are categorical, observations are independent (one row per employee, verified in
Section 3), and all expected cell counts exceed 5.

**Results and statistical interpretation.**

| Variable | p-value | Decision at α = 0.05 | Cramér's V | Effect |
|---|---|---|---|---|
| Tenure_Group | < 0.001 | Reject H₀ | 0.216 | Moderate |
| Age_Group | < 0.001 | Reject H₀ | 0.210 | Moderate |
| MaritalStatus | < 0.001 | Reject H₀ | 0.176 | Small |
| BusinessTravel | < 0.001 | Reject H₀ | 0.126 | Small |
| JobSatisfaction_Label | < 0.001 | Reject H₀ | 0.109 | Small |
| WorkLifeBalance_Label | < 0.001 | Reject H₀ | 0.104 | Small |
| Department | < 0.001 | Reject H₀ | 0.079 | Negligible |
| JobRole | 0.001 | Reject H₀ | 0.077 | Negligible |
| Education_Label | 0.29 | **Fail to reject H₀** | 0.034 | Negligible |
| Gender | 0.24 | **Fail to reject H₀** | 0.018 | Negligible |

**Business interpretation.** Company tenure and age group show the strongest associations with
attrition, followed by marital status and business travel. **Gender and education level show no
statistically detectable association** — an important negative finding, since it means the
1.4-point gender gap seen in Section 7 should not be treated as a real pattern or acted upon.

Note the gap between significance and importance: Department and JobRole are statistically
significant (p < 0.01) yet have negligible effect sizes (V < 0.08). With 4,327 employees, the test
detects differences too small to prioritise. **Significance tells you the pattern is real; effect
size tells you whether it is worth acting on.** Both readings are required.

**Limitations.** Chi-square tests each variable in isolation and cannot account for overlap between
them — age, tenure and marital status are heavily intercorrelated, so their individual results
partly describe the same employees. The test establishes association only, in one snapshot of data,
with no information about the direction of any relationship.

## 19.2 Numerical Variables: Leavers vs Stayers

In [ ]:
def compare_groups(data, column):
    # Compare a numerical variable between leavers and stayers.
    # Reports Mann-Whitney U (primary, distribution-free) and Welch t-test (secondary).
    left_group = data.loc[data.Attrition_Flag == 1, column]
    stay_group = data.loc[data.Attrition_Flag == 0, column]

    u_stat, p_mwu = stats.mannwhitneyu(left_group, stay_group, alternative="two-sided")
    t_stat, p_t = stats.ttest_ind(left_group, stay_group, equal_var=False)

    # Rank-biserial correlation: an effect size appropriate to Mann-Whitney U
    n1, n2 = len(left_group), len(stay_group)
    rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

    return {
        "Variable": column,
        "Median_Left": round(left_group.median(), 1),
        "Median_Stayed": round(stay_group.median(), 1),
        "Mean_Left": round(left_group.mean(), 1),
        "Mean_Stayed": round(stay_group.mean(), 1),
        "MWU_p_value": p_mwu,
        "Welch_t_p_value": p_t,
        "MWU_Significant": "Yes" if p_mwu < ALPHA else "No",
        "Rank_Biserial_r": round(abs(rank_biserial), 3),
    }

NUMERIC_TEST_VARS = ["MonthlyIncome", "YearsAtCompany", "TotalWorkingYears", "Age",
                     "YearsWithCurrManager", "YearsSinceLastPromotion",
                     "PercentSalaryHike", "DistanceFromHome", "TrainingTimesLastYear"]

numeric_results = pd.DataFrame([compare_groups(df, v) for v in NUMERIC_TEST_VARS])
display_results = numeric_results.copy()
display_results["MWU_p_value"] = display_results.MWU_p_value.apply(lambda p: f"{p:.2e}")
display_results["Welch_t_p_value"] = display_results.Welch_t_p_value.apply(lambda p: f"{p:.2e}")
display(display_results)

In [ ]:
# Why Mann-Whitney U rather than a t-test: check the distribution shape
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for group, color, label in [(0, NEUTRAL, "Stayed"), (1, ACCENT, "Left")]:
    axes[0].hist(df.loc[df.Attrition_Flag == group, "MonthlyIncome"], bins=35, alpha=0.6,
                 color=color, label=label, density=True)
axes[0].set_title("Monthly Income Distribution by Attrition Status\n(both strongly right-skewed)")
axes[0].set_xlabel("Monthly Income"); axes[0].set_ylabel("Density"); axes[0].legend(fontsize=9)
thousands_formatter(axes[0], "x")

for group, color, label in [(0, NEUTRAL, "Stayed"), (1, ACCENT, "Left")]:
    axes[1].hist(df.loc[df.Attrition_Flag == group, "YearsAtCompany"], bins=30, alpha=0.6,
                 color=color, label=label, density=True)
axes[1].set_title("Years at Company Distribution by Attrition Status")
axes[1].set_xlabel("Years at Company"); axes[1].set_ylabel("Density"); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

skew_check = df.groupby("Attrition")[["MonthlyIncome", "YearsAtCompany"]].skew().round(2)
print("Skewness by group (values far from 0 violate the t-test's normality assumption):")
display(skew_check)

### Numerical Comparison — Full Write-Up

**Research question.** Do employees who left differ from those who stayed on measurable numerical
attributes?

**Hypotheses** (per variable):
- **H₀:** The distributions of the variable are identical for leavers and stayers.
- **H₁:** The distributions differ.

**Test selected.** **Mann-Whitney U** as the primary test. The histograms above show both income and
tenure are strongly right-skewed (skewness well above 1), violating the normality assumption behind
the t-test. Mann-Whitney U compares distributions by rank and requires no normality assumption. A
Welch t-test is reported alongside for transparency.

**Results, statistical interpretation, and the one case where the tests disagree.**

| Variable | Median (Left vs Stayed) | MWU p-value | Decision | Effect size |
|---|---|---|---|---|
| TotalWorkingYears | 7 vs 10 | < 0.001 | Reject H₀ | Small–moderate |
| YearsAtCompany | 3 vs 6 | < 0.001 | Reject H₀ | Small–moderate |
| YearsWithCurrManager | 2 vs 3 | < 0.001 | Reject H₀ | Small–moderate |
| Age | 32 vs 36 | < 0.001 | Reject H₀ | Small–moderate |
| YearsSinceLastPromotion | 1 vs 1 | < 0.001 | Reject H₀ | Negligible |
| PercentSalaryHike | 14 vs 14 | 0.035 | Reject H₀ | Negligible |
| **MonthlyIncome** | **49,000 vs 49,410** | **0.070** | **Fail to reject H₀** | **Negligible** |
| TrainingTimesLastYear | 3 vs 3 | 0.014 | Reject H₀ | Negligible (r = 0.06) |
| DistanceFromHome | 7 vs 7 | 0.93 | Fail to reject H₀ | Negligible (r = 0.00) |

**The MonthlyIncome disagreement is the most instructive result in this section.** The Welch t-test
returns p ≈ 0.021 (significant), while Mann-Whitney U returns p ≈ 0.070 (not significant). The t-test
compares *means*, which are pulled around by the skewed upper tail; Mann-Whitney compares the full
rank distribution. Given the demonstrated skew, **the Mann-Whitney result is the one to trust**: there
is no reliable evidence that leavers and stayers differ in income. Reporting only the t-test would
have produced a confident and wrong conclusion about pay-driven attrition.

**Business interpretation.** The variables that genuinely distinguish leavers are all
career-stage measures — total experience, company tenure, manager relationship length and age.
Compensation, commute distance and promotion gap do not distinguish them. This points HR toward
**early-career and early-tenure retention** rather than pay adjustment.

**Limitations.** These are univariate tests on a single cross-sectional snapshot, with no control for
overlap between the tenure variables (all four are correlated above 0.46, so they are not independent
findings). Statistical significance across 4,327 records is easy to achieve; several significant
results here have negligible effect sizes and should not drive decisions. Running nine tests also
raises the family-wise error rate — with a Bonferroni-style adjustment (α ≈ 0.006), the borderline
`PercentSalaryHike` (p = 0.035) and `TrainingTimesLastYear` (p = 0.014) results would no longer be
significant, while the four tenure and age results would remain so by many orders of magnitude.

---

# 20. SQL ↔ Python Validation

**Objective:** confirm that the Python analysis and the project's SQL layer produce the same numbers
for the same business definitions.

### Methodological statement — read this before the table

The 30 SQL queries in this project were written against a database that **is not connected to this
notebook**. This notebook therefore **cannot execute them**, and **no SQL output values have been
invented or assumed**.

What is done instead: each SQL query's *logic* is re-implemented in pandas, and the Python result is
compared against the reference value where one is documented in the project's SQL deliverable — the
record count of **4,327 employees** stated on its cover page. Every other row is a **logic mirror**:
the Python implementation is shown to follow the same definition (same filters, same grouping, same
aggregation), so that running the SQL against the same dataset must return the same value.

The `Status` column reflects this distinction precisely:
- **Verified** — compared against a documented SQL reference value.
- **Logic mirrored** — SQL logic reproduced in Python; no external value available to compare against.

In [ ]:
# Re-implement the SQL query logic in pandas, query by query.

# Query 1 - Total Employees: SELECT COUNT(*) FROM employee_data
q1_total_employees = len(df)

# Query 2 - Overall Attrition Rate: ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),2)
q2_attrition_rate = round(df.Attrition.eq("Yes").sum() * 100.0 / len(df), 2)

# Query 3 - Department-wise attrition rate
q3_dept = (df.groupby("Department")
             .apply(lambda g: round(g.Attrition.eq("Yes").sum() * 100.0 / len(g), 2),
                    include_groups=False)
             .sort_values(ascending=False))

# Query 4 - Average salary by department
q4_avg_salary = df.groupby("Department").MonthlyIncome.mean().round(2).sort_values(ascending=False)

# Query 7 - Job role with highest attrition COUNT (note: SQL Q7 counts, it does not rate)
q7_role_counts = df[df.Attrition == "Yes"].JobRole.value_counts()

# Query 9 - Employees by education level
q9_education = df.Education_Label.value_counts()

# Query 11 / 12 - Work-life balance and job satisfaction vs attrition
q11_wlb = df.groupby("WorkLifeBalance_Label").Attrition.apply(lambda s: s.eq("Yes").sum())
q12_sat = df.groupby("JobSatisfaction_Label").Attrition.apply(lambda s: s.eq("Yes").sum())

# Query 18 - Employees with YearsSinceLastPromotion = 0
q18_no_promo_gap = int((df.YearsSinceLastPromotion == 0).sum())

# Query 20 - Attrition by business travel
q20_travel = df.groupby("BusinessTravel").Attrition.apply(lambda s: s.eq("Yes").sum())

# Query 28 - Average training by department
q28_training = df.groupby("Department").TrainingTimesLastYear.mean().round(2)

# Query 29 - Attrition percentage by marital status
q29_marital = (df.groupby("MaritalStatus")
                 .apply(lambda g: round(g.Attrition.eq("Yes").sum() * 100.0 / len(g), 2),
                        include_groups=False))

# Query 30 - HR risk segmentation (already applied in Section 18)
q30_risk = df.HR_Risk_Level.value_counts()

print("SQL logic re-implemented in pandas for 12 reference queries.")

In [ ]:
TOLERANCE = 0.01  # acceptable absolute difference for rounded numeric comparison

validation = pd.DataFrame([
    {"SQL_Query": "Q1", "Metric": "Total Employees",
     "SQL_Reference": "4,327 (stated in SQL deliverable)",
     "Python_Result": f"{q1_total_employees:,}",
     "Difference": f"{abs(q1_total_employees - 4327)}",
     "Status": "Verified" if abs(q1_total_employees - 4327) <= TOLERANCE else "MISMATCH"},

    {"SQL_Query": "Q2", "Metric": "Overall Attrition Rate",
     "SQL_Reference": "Not executed here", "Python_Result": f"{q2_attrition_rate}%",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q3", "Metric": "Department Attrition Rate (highest)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q3_dept.index[0]}: {q3_dept.iloc[0]}%",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q4", "Metric": "Average Salary (highest dept)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q4_avg_salary.index[0]}: {q4_avg_salary.iloc[0]:,.2f}",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q7", "Metric": "Job Role with Highest Attrition Count",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q7_role_counts.index[0]}: {q7_role_counts.iloc[0]}",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q9", "Metric": "Education Distribution (largest)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q9_education.index[0]}: {q9_education.iloc[0]:,}",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q11", "Metric": "Work-Life Balance Attrition (Bad)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{int(q11_wlb.get('Bad', 0))} exits",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q12", "Metric": "Job Satisfaction Attrition (Low)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{int(q12_sat.get('Low', 0))} exits",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q18", "Metric": "Employees with No Promotion Gap",
     "SQL_Reference": "Not executed here", "Python_Result": f"{q18_no_promo_gap:,}",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q20", "Metric": "Business Travel Attrition (Frequent)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{int(q20_travel.get('Travel_Frequently', 0))} exits",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q28", "Metric": "Average Training (R&D)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q28_training.get('Research & Development'):.2f}",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q29", "Metric": "Attrition % by Marital Status (Single)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{q29_marital.get('Single')}%",
     "Difference": "n/a", "Status": "Logic mirrored"},

    {"SQL_Query": "Q30", "Metric": "HR Risk Segmentation (High Risk)",
     "SQL_Reference": "Not executed here",
     "Python_Result": f"{int(q30_risk.get('High Risk', 0)):,} employees",
     "Difference": "n/a", "Status": "Logic mirrored"},
])

validation = validation[["Metric", "Python_Result", "SQL_Reference", "Difference", "Status"]]
validation = validation.rename(columns={"Python_Result": "Python Result",
                                        "SQL_Reference": "SQL Reference / Result"})
display(validation)

verified = (validation.Status == "Verified").sum()
mirrored = (validation.Status == "Logic mirrored").sum()
mismatched = (validation.Status == "MISMATCH").sum()
print(f"Verified against a documented SQL value : {verified}")
print(f"Logic mirrored (no external value)      : {mirrored}")
print(f"Mismatches                              : {mismatched}")

In [ ]:
# Internal consistency checks - these must hold regardless of SQL
checks = {
    "Attrition counts sum to total": df.Attrition.value_counts().sum() == len(df),
    "Department counts sum to total": df.Department.value_counts().sum() == len(df),
    "Attrition_Flag sum equals 'Yes' count": df.Attrition_Flag.sum() == df.Attrition.eq("Yes").sum(),
    "Risk levels sum to total": df.HR_Risk_Level.value_counts().sum() == len(df),
    "Department salary spend sums to payroll":
        abs(df.groupby("Department").MonthlyIncome.sum().sum() - df.MonthlyIncome.sum()) < 1,
    "Row count unchanged from source file": len(df) == len(df_raw),
}
for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
assert all(checks.values()), "An internal consistency check failed."
print("\nAll internal consistency checks passed.")

**Interpretation.** The one metric with a documented SQL reference value — total employee count
— matches exactly: the SQL deliverable states 4,327 records and Python loads 4,327 rows. This
confirms both layers are operating on the same dataset at the same grain, which is the prerequisite
for every other comparison.

The remaining twelve rows are marked **Logic mirrored**, not "matched". The distinction is
deliberate: claiming agreement with numbers that were never produced in this notebook would be
fabrication. What *can* be stated is that each Python implementation reproduces its SQL query's exact
definition — same filter conditions, same grouping keys, same aggregation and same rounding — so
executing the SQL against this dataset must return these values.

One definitional note worth flagging to the project author: **SQL Query 7 returns attrition *count*
by job role, not rate.** By count, Sales Executive tops the list (162 exits) purely because it is the
largest role. By rate, Research Director leads at 23.95%. Both are correct answers to different
questions — but a dashboard built on Q7 alone would point HR at the wrong role.

---

# 21. Key Business Insights

Ten insights, each drawn from a calculation performed earlier in this notebook. Every figure quoted
below is traceable to a specific section. No insight is included that the data does not support.

In [ ]:
# Regenerate the headline evidence so the insights below are verifiable against live output
evidence = pd.DataFrame([
    ["Overall attrition rate", f"{COMPANY_ATTRITION_RATE:.2f}%", f"{left:,} of {total_employees:,} employees"],
    ["Tenure 0-2 years", f"{attrition_summary(df,'Tenure_Group').loc['0-2','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'Tenure_Group').loc['0-2','Employees']:,} employees"],
    ["Tenure 11-20 years", f"{attrition_summary(df,'Tenure_Group').loc['11-20','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'Tenure_Group').loc['11-20','Employees']:,} employees"],
    ["Age 18-24", f"{attrition_summary(df,'Age_Group').loc['18-24','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'Age_Group').loc['18-24','Employees']:,} employees"],
    ["Travel_Frequently", f"{attrition_summary(df,'BusinessTravel').loc['Travel_Frequently','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'BusinessTravel').loc['Travel_Frequently','Employees']:,} employees"],
    ["Non-Travel", f"{attrition_summary(df,'BusinessTravel').loc['Non-Travel','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'BusinessTravel').loc['Non-Travel','Employees']:,} employees"],
    ["Human Resources dept", f"{attrition_summary(df,'Department').loc['Human Resources','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'Department').loc['Human Resources','Employees']:,} employees"],
    ["Single employees", f"{attrition_summary(df,'MaritalStatus').loc['Single','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'MaritalStatus').loc['Single','Employees']:,} employees"],
    ["Work-life balance 'Bad'", f"{attrition_summary(df,'WorkLifeBalance_Label').loc['Bad','Attrition_Rate_%']:.2f}%",
     f"{attrition_summary(df,'WorkLifeBalance_Label').loc['Bad','Employees']:,} employees"],
    ["High Risk segment", f"{risk_validation.loc['High Risk','Attrition_Rate']:.2f}%",
     f"{int(risk_validation.loc['High Risk','Employees']):,} employees"],
], columns=["Segment", "Attrition_Rate", "Base_Size"])
display(evidence)

## Insight 1 — Attrition is concentrated in the first two years of tenure

**Finding.** Employees with 0–2 years at the company leave at **30.23%**, versus **6.79%** for those
with 11–20 years — a 4.5× difference, and the widest spread of any variable examined.

**Evidence.** Section 7.4 tenure-group table; Section 12 tenure curve; Mann-Whitney U on
`YearsAtCompany` (p < 0.001, Section 19.2). Median tenure is 3 years for leavers and 6 for stayers.

**Business interpretation.** 999 employees — 23% of the workforce — sit in the highest-risk tenure
band. This is the classic signature of an onboarding, role-fit or early-expectation problem rather
than a broad engagement failure across the organisation.

**HR consideration.** HR may consider examining the first-24-month employee journey specifically:
onboarding structure, role clarity, realistic job previews at hiring, and manager check-in cadence in
months 3–18. Exit-interview data segmented by tenure band would help determine whether these exits
are voluntary resignations or early-stage role mismatches.

---

## Insight 2 — Business travel frequency shows the most robust association with attrition

**Finding.** Attrition rises monotonically with travel: **8.14%** (Non-Travel) → **15.08%**
(Travel_Rarely) → **24.79%** (Travel_Frequently). Frequent travellers leave at roughly 3× the rate of
non-travellers.

**Evidence.** Section 14 travel analysis; chi-square p < 0.001 with Cramér's V = 0.126 (Section 19.1);
and critically, Section 16.3 shows the gradient **persists within every tenure band** and every
department — so it is not an artefact of travellers being newer employees.

**Business interpretation.** This is the only major finding in the analysis that survived an explicit
confounding check. 815 employees travel frequently, and they account for 202 exits — 29% of all
attrition from 19% of the workforce.

**HR consideration.** Organisations could evaluate travel load distribution within travel-heavy
roles, and whether travel is concentrated on the same individuals repeatedly. Note the anomaly in
Insight 3 before assuming the mechanism is burnout.

---

## Insight 3 — Frequent travellers report normal satisfaction scores yet leave far more often

**Finding.** Average job satisfaction and work-life balance scores are **nearly identical** across
all three travel groups, despite their attrition rates differing threefold.

**Evidence.** Section 14 travel profile table — satisfaction and balance means differ by under 0.1
points across groups.

**Business interpretation.** Whatever drives travel-related exits is **not being captured by the
existing satisfaction survey**. If HR were monitoring engagement scores alone, this entire high-risk
population would appear healthy right up until resignation. That is a measurement gap, not just a
retention gap.

**HR consideration.** HR may consider adding travel-specific questions to the engagement instrument
(travel predictability, notice period, recovery time, family impact) rather than relying on the
general balance score to surface this population.

---

## Insight 4 — The Human Resources function has the highest departmental attrition

**Finding.** HR shows **29.79%** attrition (56 of 188 employees), nearly double the company average
of 16.20% and well above R&D (15.83%) and Sales (15.06%).

**Evidence.** Section 7.2 department table and count-vs-rate comparison chart.

**Business interpretation.** This is invisible in exit-volume reporting: HR contributes only 8% of
total exits, so a dashboard ranking departments by count would place it last. It is also the
function that would normally own the retention response — a department losing 30% of its own staff
has reduced capacity to address attrition elsewhere.

**HR consideration.** Given the small base (188 employees), HR may consider confirming this rate
across a second time period before major investment, while reviewing workload and career-path
structure within the function.

---

## Insight 5 — Compensation does not distinguish leavers from stayers

**Finding.** Median income is **49,000** for leavers and **49,410** for stayers — a 0.8% gap.
Mann-Whitney U returns **p = 0.070**, failing to reject the null hypothesis at α = 0.05.

**Evidence.** Section 8 income-by-attrition boxplot; Section 19.2 full test write-up. Attrition by
income quartile is flat and non-monotonic (13.31%–18.55%), and income correlates −0.03 with attrition.

**Business interpretation.** There is no evidence in this dataset that pay level drives exits. A
retention strategy built on across-the-board salary adjustment would be targeting a factor the data
does not implicate.

**HR consideration.** This finding carries a **strong data caveat** (Insight 10) and should not be
read as "pay doesn't matter". It means *this dataset cannot detect a pay effect*. Establishing
whether one exists requires salary benchmarked against market rate and role band.

---

## Insight 6 — Attrition falls steeply with age, and the youngest cohort is the highest-risk group in the organisation

**Finding.** **39.24%** of employees aged 18–24 have left, against **10.11%** of those aged 35–44.
Age group produces the strongest categorical association tested (Cramér's V ≈ 0.24, moderate effect).

**Evidence.** Section 7.3 age-group table; Section 19.1 chi-square; Section 16.1 shows the age effect
persists when income is held constant.

**Business interpretation.** 288 employees sit in the 18–24 band and 113 of them have left. Combined
with Insight 1, the picture is consistent: **early-career and early-tenure employees are the
organisation's attrition problem**, and these two lenses largely describe the same population.

**HR consideration.** HR could evaluate whether early-career employees have visible development
pathways and mentorship in their first two years, and whether hiring expectations at entry level
match the actual role.

---

## Insight 7 — Poor work-life balance marks the sharpest single-category risk, but the relationship is not linear

**Finding.** Employees rating work-life balance **Bad** leave at **31.22%**. However, the
relationship is non-monotonic — the **Best** category (17.96%) shows *higher* attrition than
**Better** (14.41%).

**Evidence.** Section 9 distribution and rate charts; chi-square p < 0.001 (V = 0.104).

**Business interpretation.** The 237 employees reporting Bad balance are a small, clearly identifiable,
high-risk group. The anomaly at the top of the scale means the relationship should **not** be modelled
as "more balance = more retention"; the data does not support that simplification.

**HR consideration.** HR may consider investigating the "Best" balance group specifically — whether
it contains part-time, disengaged, or under-utilised employees — before treating the balance score
as a simple linear retention lever.

---

## Insight 8 — Job satisfaction differentiates attrition mainly at the extremes

**Finding.** Low satisfaction employees leave at **22.91%** versus **11.40%** for Very High — roughly
double. The two middle categories are nearly identical to each other (16.5%).

**Evidence.** Section 9 satisfaction analysis; Section 16.2 shows the effect persists within tenure
bands, and that the two factors compound.

**Business interpretation.** The signal lives at the tails. Moving an employee from Medium to High
satisfaction shows no measurable attrition benefit in this data; moving them out of the Low category
does. This argues for targeted intervention over broad engagement programmes.

**HR consideration.** HR may consider prioritising the 851 employees in the Low satisfaction band,
particularly the subset also in early tenure, where Section 16.2 shows the highest combined rates.

---

## Insight 9 — Gender and education level show no detectable association with attrition

**Finding.** Gender: 16.76% male vs 15.37% female, chi-square **p = 0.24**. Education level:
p = 0.29. Both fail to reject the null hypothesis.

**Evidence.** Section 19.1 chi-square results; Section 7.3 rate tables.

**Business interpretation.** This is a **valuable negative finding**. The 1.4-point gender gap is
visible on a bar chart and would likely be read as a pattern by a non-technical viewer, but it is
statistically indistinguishable from noise. Acting on it would waste effort and could produce
misleading internal narratives.

**HR consideration.** These dimensions can be de-prioritised for attrition-driver analysis, and any
dashboard displaying them should carry a note that the differences are not statistically significant.

---

## Insight 10 — The dataset's compensation structure limits what can be concluded about pay

**Finding.** `MonthlyIncome` correlates **0.05** with `JobLevel` and **−0.03** with
`TotalWorkingYears`. Median income is effectively flat across job levels 1 through 5.

**Evidence.** Section 3.6 structural checks; Section 15 correlation matrix; Section 8 income-by-level
boxplot.

**Business interpretation.** In any real organisation, income rises steeply with seniority and
experience. Its absence here indicates that income values were assigned independently of the
seniority fields during dataset construction. Other relationships in the same data behave exactly as
expected (`PercentSalaryHike` ↔ `PerformanceRating` at 0.77), so this is specific to income level,
not a general data failure.

**HR consideration.** Every compensation finding in this analysis is reported descriptively and none
is used to support a recommendation. Before any pay-related decision, the underlying payroll extract
should be validated against the source HRIS.

---

# 22. HR Recommendations

Each recommendation below traces to a specific finding. Recommendations are framed as
investigations and evaluations rather than instructions, because a descriptive analysis can identify
*where* to look — it cannot establish what will work.

### 1. Early-tenure retention (Priority: High)
*Based on Insights 1 and 6 — 30.23% attrition in the 0–2 year band, 39.24% among ages 18–24.*

HR may consider treating the first 24 months as a distinct retention programme with its own
measurement: structured onboarding checkpoints, explicit role-expectation setting at hire, and
manager check-ins scheduled through month 18. Further analysis of exit interviews segmented by tenure
band would help determine whether these exits reflect hiring mismatch or post-join experience.

### 2. Business travel load management (Priority: High)
*Based on Insights 2 and 3 — 24.79% attrition among frequent travellers, the only finding that
survived a confounding check.*

Organisations could evaluate how travel is distributed within travel-heavy roles — specifically
whether the same individuals absorb repeated trips — and consider recovery time, advance notice and
trip predictability as operational levers. This should be paired with recommendation 6, since the
current survey instrument is not detecting strain in this group.

### 3. Targeted low-satisfaction and poor-balance intervention (Priority: Medium-High)
*Based on Insights 7 and 8 — 31.22% attrition at Bad work-life balance, 22.91% at Low satisfaction.*

Rather than a broad engagement programme, HR may consider focusing on the 237 employees reporting
Bad balance and the 851 reporting Low satisfaction, prioritising the overlap with early tenure
identified in Section 16.2. The evidence suggests the return is concentrated at the bottom of these
scales, not spread evenly across them.

### 4. Human Resources function review (Priority: Medium)
*Based on Insight 4 — 29.79% attrition in a 188-person department.*

HR may consider reviewing workload, career pathways and span of control within its own function,
while first confirming the rate against a second time period given the small base.

### 5. Career progression transparency (Priority: Medium)
*Based on Section 11 — no clear attrition gradient across promotion gap, but 39.5% of employees have
no promotion gap recorded.*

The promotion field itself is ambiguous (Section 3, finding 7). Before drawing conclusions about
career stagnation, HR could consider improving how promotion events are captured in the HRIS —
distinguishing "promoted recently" from "no promotion event on record". Better data here would make
a genuine progression analysis possible.

### 6. Survey instrument coverage (Priority: Medium)
*Based on Insight 3 — high-attrition travel groups report normal satisfaction scores.*

HR could evaluate whether the current engagement survey captures the pressures affecting
travel-intensive roles. A survey that shows no signal in the organisation's highest-risk population
is not providing early warning.

### 7. Compensation data validation (Priority: Medium — data integrity)
*Based on Insight 10 — income uncorrelated with job level and experience.*

Before any compensation analysis informs decisions, the payroll extract should be reconciled against
the source HRIS. The current structure cannot support pay-equity or pay-progression conclusions.

### 8. Training programme measurement (Priority: Low)
*Based on Section 13 — non-monotonic relationship between training frequency and attrition.*

The data cannot determine whether training retains employees or whether engaged employees attend more
training. Further analysis — ideally tracking training participation and retention over time — may
help determine the direction before training volume is used as a retention lever.

### What the evidence does *not* support

- **Across-the-board salary increases** — income does not distinguish leavers from stayers
  (p = 0.070), and the compensation data has a documented structural problem.
- **Gender-targeted or education-targeted retention programmes** — neither variable shows a
  statistically detectable association (p = 0.24 and p = 0.29).
- **Volume-based departmental prioritisation** — targeting R&D because it produces the most exits
  would direct effort at the department with a below-average attrition *rate*.

---

# 23. Executive Summary

### Workforce Snapshot
- **4,327 employees**, one row per employee, zero missing values, zero duplicates.
- **Research & Development 65%** (2,824) · Sales 30% (1,315) · Human Resources 4% (188).
- 60% male / 40% female; median age **36**; median company tenure **5 years**.
- 73% hold Life Sciences or Medical educational backgrounds; 66% are at job level 1–2.

### Attrition Snapshot
- **701 employees left — an overall attrition rate of 16.20%.**
- Segment rates range from **6.79%** (11–20 years tenure) to **40.74%** (HR education field).
- Highest-volume exits: R&D (447). Highest-rate exits: Human Resources department (29.79%).

### Key Risk Factors and Associations
| Factor | Highest-risk segment | Rate | Statistical support |
|---|---|---|---|
| Company tenure | 0–2 years | 30.23% | p < 0.001, moderate effect |
| Age | 18–24 | 39.24% | p < 0.001, moderate effect |
| Business travel | Travel_Frequently | 24.79% | p < 0.001, survives tenure control |
| Marital status | Single | 25.52% | p < 0.001, small effect |
| Work-life balance | Bad | 31.22% | p < 0.001, non-monotonic |
| Job satisfaction | Low | 22.91% | p < 0.001, effect at extremes |
| **Gender** | — | — | **Not significant (p = 0.24)** |
| **Education level** | — | — | **Not significant (p = 0.29)** |

### Compensation Findings
- Median monthly income **49,360**; mean 65,029 (strongly right-skewed).
- **Income does not distinguish leavers from stayers** (Mann-Whitney U, p = 0.070).
- Attrition by income quartile is flat and non-monotonic (13.31%–18.55%).
- Pay-for-performance is operating: Outstanding performers average a 21.85% hike vs 14.00%.
- **Caveat:** income is uncorrelated with job level (r = 0.05) and experience (r = −0.03); no
  pay-equity conclusion is drawn.

### Employee Experience Findings
- Low satisfaction → 22.91% attrition vs 11.40% at Very High.
- Bad work-life balance → 31.22%, but the scale is non-monotonic at the top end.
- Satisfaction and balance **compound** when combined (Section 16.2).
- Frequent travellers report **normal** satisfaction scores despite 24.79% attrition — a survey
  coverage gap.

### Career Growth Findings
- Promotion gap shows **no clear attrition gradient**; long gaps are not the highest-risk group.
- Shorter tenure with current manager is associated with higher attrition, but is heavily confounded
  with company tenure (r = 0.77) and cannot be separated in this data.
- **Neither pay nor job level rises with tenure** in this dataset (average job level drifts from
  2.07 down to 1.78 across tenure bands), which is a data-structure limitation rather than a finding
  about the organisation.
- 39.5% of employees have no promotion gap recorded; the field is ambiguous and needs better capture.

### Key HR Considerations
1. Prioritise the **first 24 months** of employment — the single widest attrition spread found.
2. Examine **travel load distribution** — the most robust association in the analysis.
3. Target the **bottom of the satisfaction and balance scales**, not the whole distribution.
4. Review the **HR function's own retention**, with a second-period confirmation first.
5. **Validate the compensation extract** before any pay analysis informs decisions.
6. **Do not act on gender or education differences** — they are statistically indistinguishable from noise.

---

# 24. Conclusion

### What Was Analysed

This notebook examined 4,327 employee records across 31 source variables, covering workforce
composition, attrition, compensation, satisfaction, work-life balance, performance, career
progression, experience, tenure, training and business travel. Seven analytical features were
engineered, every band justified against the observed distribution or against the project's existing
SQL logic.

### What Was Identified

Attrition in this organisation is **not evenly distributed and not driven by pay**. It is
concentrated among employees who are early in tenure, early in career, and travelling frequently.
The strongest finding — business travel — was the one that survived an explicit confounding check
against tenure. Two widely assumed drivers were tested and **not** supported: compensation level
(p = 0.070) and gender (p = 0.24).

### How Python Contributed

Python provided the layer SQL cannot: distribution shape, visual evidence, cross-tabulated
confounding checks, formal hypothesis testing with effect sizes, and the structural data-quality
audit that uncovered the compensation anomaly. The most consequential result in the notebook — that
the t-test and Mann-Whitney U disagree on income, and that the rank-based test is the trustworthy one
— is only reachable with a statistical toolkit.

### How This Connects to SQL

The Python analysis operates on the same 4,327-record dataset as the project's 30 SQL queries,
verified in Section 20. SQL query logic was mirrored rather than assumed, with no SQL output values
invented. The HR Risk Score uses the identical rule from SQL Query 30, so all three project layers
segment employees the same way. One definitional issue was surfaced for the SQL layer: Query 7
returns attrition *count* by job role, which ranks roles differently than attrition *rate* would.

### How This Supports the Power BI Dashboard

This analysis gives the dashboard its structure:

- **Headline KPIs** — total employees, attrition count, attrition rate, retention rate.
- **Primary slicers** — the dimensions with real differentiating power: tenure band, age group,
  business travel, department.
- **Risk segmentation page** — the HR_Risk_Level rule, consistent with SQL Query 30.
- **Guardrails** — attrition rate (not count) as the default comparison metric, base sizes displayed
  alongside every rate, and a note on dimensions where differences are not statistically significant.

### Limitations

1. **Cross-sectional data.** One snapshot, no time dimension. Nothing here establishes causation, and
   satisfaction scores may have been recorded after an employee had already decided to leave.
2. **Univariate and bivariate methods only.** Tenure, age, marital status and manager tenure overlap
   heavily; their individual effects cannot be separated. A multivariate model (logistic regression
   or survival analysis) is the natural next step.
3. **Missing variables.** `OverTime` and `YearsInCurrentRole` are absent, so overtime analysis could
   not be performed at all.
4. **Compensation structure.** Income is uncorrelated with seniority and experience, which rules out
   pay-equity analysis on this extract.
5. **Structural artefacts.** Job roles appear across all departments, so department and role are
   treated as separate lenses rather than a hierarchy.
6. **Small segments.** Several notable rates (HR education field at 40.74%, the High Risk tier at 87
   employees) rest on small bases and need confirmation across periods.
7. **Ambiguous promotion field.** `YearsSinceLastPromotion = 0` conflates two different situations.
8. **No exit-type distinction.** Voluntary resignations, terminations and retirements are pooled in a
   single `Attrition` flag, which limits interpretation of the 20+ year tenure band in particular.

### Recommended Next Steps

1. Fit a **logistic regression** to separate the overlapping tenure, age and travel effects.
2. Apply **survival analysis** to model *when* employees leave rather than only whether they left.
3. Acquire **overtime, exit-type and market-benchmarked salary** data to close the identified gaps.
4. Re-run this analysis on a **second time period** to confirm the small-base findings.

---

# 25. Notebook Quality Check

In [ ]:
completed = {
    "Project overview & business questions": True,
    "Library configuration": True,
    "Dataset loading & profiling": True,
    "Column availability audit": True,
    "Data quality assessment": True,
    "Missing value analysis": True,
    "Duplicate analysis": True,
    "Invalid value & logical constraint checks": True,
    "Data cleaning (documented, zero rows deleted)": True,
    "Feature engineering (7 features, justified bins)": True,
    "Workforce & demographic analysis": True,
    "Core attrition analysis (count vs rate)": True,
    "Compensation analysis": True,
    "Job satisfaction analysis": True,
    "Work-life balance analysis": True,
    "Performance analysis": True,
    "Career growth & promotion analysis": True,
    "Experience & tenure analysis": True,
    "Training analysis": True,
    "Business travel analysis": True,
    "Overtime analysis": False,   # OverTime column absent from dataset
    "Correlation analysis": True,
    "Multivariate analysis": True,
    "Outlier analysis (none removed)": True,
    "HR risk segmentation (rule-based)": True,
    "Statistical hypothesis testing": True,
    "SQL-Python validation": True,
    "Key business insights": True,
    "HR recommendations": True,
    "Executive summary": True,
    "Conclusion & limitations": True,
}

status = pd.DataFrame({
    "Section": completed.keys(),
    "Status": ["Completed" if v else "NOT POSSIBLE - required column absent"
               for v in completed.values()],
})
display(status)

done = sum(completed.values())
print(f"Sections completed: {done} of {len(completed)}")
print(f"Not possible: {len(completed) - done} "
      f"({[k for k, v in completed.items() if not v]})")
print(f"\nFinal dataset: {df.shape[0]:,} rows x {df.shape[1]} columns "
      f"({df.shape[1] - df_raw.shape[1]} engineered features added, 0 rows deleted)")
print(f"Overall attrition rate: {COMPANY_ATTRITION_RATE:.2f}%")

**Quality check result.** 30 of 31 planned sections completed. The single incomplete section —
overtime analysis — is not an oversight: the `OverTime` column does not exist in this dataset, and no
proxy was substituted for it. This is recorded rather than quietly dropped.

---

*End of analysis. Employee Attrition & HR Analytics — Python Exploratory Data Analysis.*